# BDC2026 Satria Data — Waste Image Classification (SigLIP multi-backbone frozen-probe, best version)

**Notebook conversion notice.** This notebook is a *documentation-friendly restructuring* of the best
single-cell pipeline (`satdat_siglip.ipynb`) into ordered, narrated cells, **plus** a complete
Exploratory Data Analysis (section 3) and Advanced Post-processing Analysis (sections 7 & 10).

**Guarantee of identical output.** Every model-pipeline cell (sections **1, 2, 4, 5, 6, 8, 9**) is copied
**byte-for-byte** from the original single cell — the source string was sliced at its own section banners
and re-assembled with *zero edits*; a hash check confirms the concatenation equals the original. Nothing was
reordered *within* the pipeline, reparametrized, or re-seeded. Therefore `oof`, `test_probs`,
`decision_weights`, and **`submission.csv` are bit-for-bit identical** to running the original notebook.

**What the NEW sections may and may not do.** Sections **3, 7, 10** are *new, read-only* analysis. They only
*read* `df`, image files, cached features, and the OOF / test arrays the pipeline already produced. They use
**isolated random state** (`random_state=SEED`, `np.random.default_rng(SEED)`) and never touch the global
torch RNG that training depends on, never write back into `df` / folds / features / probabilities, and never
retrain a head. All model-producing sections keep their exact original relative order, so the analysis cannot
change model weights, OOF predictions, or the submission.

**Pipeline specifics carried over verbatim.** Frozen DINOv2 ViT-L/14+reg @336 (CLS+masked-mean patch,
L2-norm) linear+MLP dual head, 5-fold head ensemble; extra frozen backbones `convnext_small` and
`siglip2_l` (the SigLIP backbone, on probation); OOF-only ensemble selection (weight-tune vs logistic-meta),
per-class bias, similarity-gated kNN retrieval, temperature calibration, and the train↔test exact-dup
leakage override. All tuning is OOF-derived — no test labels are used.


## 1. Setup

Imports, global configuration (backbone choice, image size, augmentation switches, hyperparameters, seeds),
and environment/device setup. **Verbatim from the original single cell — do not modify.**


In [ ]:
# BDC2026 Satria Data — REVISED baseline v3  (target: <1h end-to-end on Kaggle GPU T4 x2)
# =============================================================================================
# WHAT CHANGED vs the previous v2(2) baseline, and WHY (the "kelemahan" fixes):
#
#   [MODEL — biggest F1 levers, all cheap because the backbone stays FROZEN]
#   (1) Backbone: vit_small_patch14_dinov2  ->  vit_LARGE_patch14_REG4_dinov2  (DINOv2 + registers).
#       Registers (Darcet et al. 2023, "ViTs Need Registers") remove artifact tokens => cleaner
#       features at the same inference cost class. This is the "better DINOv2 variant" upgrade.
#   (2) Resolution: 224 -> 336 (14*24, divisible by patch14). DINOv2 is pretrained at 518; 224
#       throws away detail on small in-frame objects. 336 is the single biggest frozen-feature lever.
#   (3) Feature = concat[ CLS token , mean(patch tokens) ]  instead of CLS/pooled only. This is the
#       standard DINOv2 linear-probe recipe and is essentially free (one forward_features call).
#   (4) Per-sample L2-normalization of each feature block before concat (leak-free, no fitting) +
#       LayerNorm inside the head => stable linear probe.
#   (5) Test-time augmentation (hflip) added to the DINOv2 path (previously only convnext had TTA).
#   (6) Head = LayerNorm -> (linear probe  +  small MLP) logit-ensemble; 5-fold head ensemble on top.
#   (7) Preprocessing informed by EDA: pad-to-square (edge-replicate) instead of center-crop, so the
#       long edge of high-aspect-ratio photos is not clipped (EDA-2 measured aspect_ratio std ~0.4).
#   (8) Multi-GPU: feature extraction sharded across both T4s via nn.DataParallel (autocast inside
#       the model so it propagates to DP worker threads).
#
#   [EDA — now actually drives decisions + surfaces label noise]
#   (9) EDA prints an explicit preprocessing recommendation; adds brightness/contrast per class.
#  (10) After CV, an OOF-disagreement pass lists the most-confidently-wrong TRAIN images per class
#       (likely mislabels) and saves suspect_labels.csv for manual review. Optional drop+retrain
#       is behind CLEAN_RETRAIN (default False — safe/reproducible; head retrain is ~seconds).
#
#   [ROBUSTNESS]
#  (11) Test file paths are globbed by id (not hardcoded ".jpg") so a non-jpg test file won't crash.
#  (12) Inference is driven by the official template CSV length (works whether test is 1000 or 1458).
#  (13) Dedup-aware CV (StratifiedGroupKFold on near-dup clusters) kept for HONEST OOF, but the
#       verbose P0/P2 audit prints are compressed to a short summary.
#
# NOTE: aHash-based near-dup grouping only makes OOF/threshold-tuning honest; it does not touch the
# test model. Manual test labels are NOT here; score locally after download.
# =============================================================================================

import os, sys, gc, math, time, random, glob, shutil, zipfile, subprocess, hashlib
import numpy as np, pandas as pd

# ============================================================
# 0. CONFIG
# ============================================================
MODE = "dinov2_linear"     # "dinov2_linear" (enhanced frozen probe, DEFAULT) | "convnext" (kept, untouched)
ENSEMBLE_WITH_CONVNEXT = False   # if True: also run convnext path and average logits/softmax with dinov2 at inference (fix D)

# --- shared ---
N_FOLDS = 5
SEED = 42
NUM_CLASSES = 3
CLASS_NAMES = {0: "Recyclable", 1: "Electronic", 2: "Organic"}
WORKERS = 4                 # Kaggle T4 x2 gives ~4 vCPU

FOLDS_TO_RUN_DINOV2 = list(range(N_FOLDS))   # head training is ~seconds/fold -> run all folds
FOLDS_TO_RUN_CONVNEXT = [0, 1]               # convnext fine-tune stays trimmed (only if MODE=convnext)

# --- DINOv2 (frozen extractor) config ---------------------------------------------------------
# Chosen for GPU T4 x2 within <1h. To scale DOWN (single T4 / P100 / time pressure) switch to:
#   DINOV2_BACKBONE = "vit_base_patch14_reg4_dinov2.lvd142m"  and/or  IMG = 224
# To scale UP (if you have lots of headroom): vit_giant_patch14_reg4_dinov2.lvd142m or IMG=448/518.
DINOV2_BACKBONE = "vit_large_patch14_reg4_dinov2.lvd142m"   # ViT-L/14 + 4 registers, embed=1024 -> feat=2048
IMG = 336                  # 14*24; DINOv2 native is 518, so 336 keeps far more detail than 224
PREPROCESS = "pad"         # "pad" (square pad, edge-replicate — recommended), "squash", or "crop"
USE_TTA = True             # hflip test-time augmentation on the test set
TTA_SCALES = [1.0, 0.9, 1.1]   # multi-scale TTA (fix H): 1.0 = base IMG, others resize the square canvas by this factor before center-cropping/padding back to IMG. vflip is NOT used (flips object gravity/orientation meaning).
BATCH_EXTRACT = 128        # total batch for extraction (split across GPUs by DataParallel)

DINOV2_HEAD_EPOCHS = 40    # head-only training over cached features is trivially cheap
DINOV2_HEAD_LR = 2e-3
DINOV2_HEAD_HIDDEN = 512   # MLP branch width (linear-probe branch is always on); 0 disables the MLP branch
BATCH_HEAD = 512
HEAD_DROPOUT = 0.2
HEAD_WD = 1e-4

# --- head hyperparameter sweep (fix E): cheap because it trains only on cached features -------
RUN_HEAD_HPO = True
HEAD_HPO_TRIALS = 24
# HPO no longer sweeps on a hardcoded CV fold (fold 0 was degenerate last run — 1 image — making the
# sweep meaningless: both backbones "found" the same cfg with f1=1.0000 on a single sample). head_hpo()
# now carves its OWN well-sized, group-aware holdout (~1/6 of data) independent of the CV folds.
HEAD_HPO_GRID = {
    "lr": [1e-3, 2e-3, 4e-3, 8e-3],
    "hidden": [256, 512, 768],
    "dropout": [0.1, 0.2, 0.3],
    "wd": [1e-5, 1e-4, 1e-3],
}

# --- feature-space mixup for head training (fix A / cheapest-highest-payoff item) --------------
USE_FEATURE_MIXUP = True
FEAT_MIXUP_ALPHA = 0.2
FEAT_MIXUP_PROB = 0.5

# --- TRAIN-TIME image augmentation for the frozen probe (the real val->test gap lever) ----------
# THE gap diagnosis: OOF=99.5 vs manual-test=98.1 and Recyclable recall collapses 99%->95% on test.
# Part of that is OOF leakage (fixed by the dedup rewrite above), but a frozen linear probe ALSO never
# sees any augmentation — the head is trained on ONE clean view per image, so it never learns to be
# invariant to the lighting / white-balance / scale / crop variation that a different camera (the test
# set) introduces. Fix: extract features from a few AUGMENTED views of each train image and train the
# head on clean+augmented together. Views are cached (frozen into feat_cache on first extraction), so
# re-runs pay nothing. Cost on a COLD run = TRAIN_AUG_VIEWS extra extraction passes for each listed
# backbone (DINOv2-L clean pass ~8-9min, so 2 views ~= +17min) -> kept to dinov2 only by default to
# protect the <1h budget; drop TRAIN_AUG_VIEWS to 1 (or [] backbones) if your GPU quota is tight.
USE_TRAIN_FEATURE_AUG = True
TRAIN_AUG_VIEWS = 2                       # number of augmented feature passes over the train set per backbone
TRAIN_AUG_BACKBONES = ["dinov2"]         # which backbones get train-aug (e.g. add "convnext_small" for a polish run)

# --- partial backbone unfreeze (fix F — biggest ceiling lever, but NOT cheap: turns the single
# frozen forward pass into ~(BACKBONE_FT_EPOCHS+1)x forward passes over the train set plus backward
# through the unfrozen blocks. Measured cost on this dataset: ~+60min (20->80min) for +0.17 OOF F1
# (98.30->98.47). Default OFF for fast iteration; flip to 2 for a final/expensive polish run only
# when Kaggle quota allows and you've confirmed the gain holds up on a controlled ablation. ---------
UNFREEZE_LAST_N_BLOCKS = 0     # 0 disables (fast default); unfreezes the last N transformer blocks + final norm
BACKBONE_FT_EPOCHS = 4         # short fine-tune once unfrozen (backbone forward+backward is not free anymore)
BACKBONE_LR = 2e-6             # discriminative LR: much smaller than HEAD lr
BACKBONE_FT_BATCH = 64

# --- masked / attention pooling for padded images (fix B) ---------------------------------------
# NOTE: "attn" mode's query vector is only ever trained inside run_backbone_finetune() (fix F) —
# it needs gradient signal from *somewhere*, and that's the only place we run multiple forward/backward
# passes over the backbone. With UNFREEZE_LAST_N_BLOCKS=0 that pass never runs, so the query would stay
# randomly-initialized. run_mode_dinov2() auto-falls-back to "masked_mean" and prints a warning if you
# select "attn" without also enabling the unfreeze — but prefer setting this explicitly to avoid relying
# on the fallback. "masked_mean" gets the same padding-dilution fix (fix B) for zero extra training cost.
POOLING = "masked_mean"    # "mean" (old behavior) | "masked_mean" (exclude padded patch tokens, cheap+safe default) | "attn" (learned query, needs UNFREEZE_LAST_N_BLOCKS>0 to train)

# --- calibration before mislabel flag / weight tuning (fix G) ----------------------------------
RUN_TEMP_SCALING = True

# --- train<->test near-dup / leakage check via cosine similarity on cached features (fix C) -----
RUN_TRAIN_TEST_DEDUP_CHECK = True
TRAIN_TEST_DUP_COS_TH = 0.98

# --- near-duplicate detection / dedup-aware CV split (for HONEST OOF only) ---------------------
# HARD LESSON from the last run: 8x8 *average*-hash + Hamming<=4 was catastrophic on this dataset --
# 188,780 "near-dup" pairs on 26k imgs, chaining via union-find into ONE 12,087-img cluster (46% of
# data) that StratifiedGroupKFold then had to dump into a single fold -> fold sizes [1, 4545, 4735,
# 12087, 4729]. Average-hash collapses each image to "which cells beat the mean brightness", so the
# many trash photos sharing plain/similar backgrounds collide as false duplicates. Fixes below:
#   (1) proper DCT-based pHash (perceptual) instead of average-hash -> far more discriminative;
#   (2) tighter Hamming + more LSH blocks for reliable recall at that threshold;
#   (3) a group-size GUARD that dissolves any oversized cluster (a hash artifact) back into singletons,
#       guaranteeing balanced folds even if the hash still over-merges a few.
RUN_DEDUP = True
DEDUP_PHASH_SIZE = 8          # low-freq DCT block kept = 8x8 -> 64-bit hash
DEDUP_DCT_FACTOR = 4          # resize to (PHASH_SIZE*FACTOR)=32px before DCT, keep top-left 8x8 low freqs
DEDUP_MAX_HAMMING = 6         # on the DCT pHash (near-dups ~<=6 bits; distinct imgs ~30+); was 4 on avg-hash
DEDUP_LSH_BLOCKS = 8          # 8 blocks of 8 bits: any pair within Hamming<=7 shares >=1 identical block
DEDUP_MAX_GROUP_FRAC = 0.01   # any dup-cluster > this fraction of the dataset (~260 imgs) is treated as a
                             # false-positive hash chain and split back into singletons (keeps folds balanced)

# --- imbalance handling (mild ~3.17:1 -> label-smoothed CE + sampler + OOF threshold tuning) ---
USE_CB_FOCAL = False
CB_BETA = 0.999
FOCAL_GAMMA = 2.0
SAMPLER_POW = 0.3 if USE_CB_FOCAL else 0.5
LS = 0.1

# --- EDA / label-noise ---
RUN_EDA = True
EDA_SAMPLE_PER_CLASS = 400
CLEAN_RETRAIN = False       # if True: drop OOF-high-confidence-wrong train rows and retrain heads once
CLEAN_CONF_TH = 0.90        # confidence threshold for the "suspected mislabel" flag

# --- convnext path config (only used if MODE=="convnext" or ENSEMBLE_WITH_CONVNEXT=True; both off by
# default -- kept in sync with EXTRA_BACKBONES' convnext_small so there's a single consistent choice) ---
CONVNEXT_BACKBONE = "convnext_small.fb_in22k_ft_in1k"
EPOCHS, LR, BB_LR_MULT, WD, WARMUP = 5, 3e-4, 0.5, 0.05, 1.0
MIXUP_A, CUTMIX_A, MIXUP_OFF = 0.2, 1.0, 1
RE_PROB, RAND_AUG, EMA_DECAY = 0.25, "rand-m7-mstd0.5-inc1", 0.999
BATCH = 64

# --- multi-backbone frozen-probe ensemble --------------------------------------------------------
# Each extra backbone runs the SAME cheap recipe as DINOv2 (frozen extract -> DualHead -> OOF), never
# fine-tuned, so wall-clock stays small (tiny/small models are much faster than ViT-L to begin with).
# kind: "token" (ViT-style forward_features returns a [B, N, C] token sequence -> reuses DINOFeat/
#       masked-mean or plain-mean pooling) | "conv" (CNN forward_features returns a [B,C,H,W] map ->
#       ConvFeat does global-average-pool). Architecturally-different backbones are a cheap diversity
#       source for the ensemble stage below (their errors tend not to correlate with the ViT probe's).
RUN_EXTRA_BACKBONES = True
# per-backbone "batch" overrides BATCH_EXTRACT: last run convnext_small extraction took 34.5min (4x the
# DINOv2-L pass despite being a smaller model at lower res!). That is NOT compute or I/O bound — at
# 12.6 img/s it sits well under the ~75 img/s dataloader ceiling; it's nn.DataParallel per-forward
# replication OVERHEAD, which dominates for a fast model. A much larger batch = far fewer forward calls
# = proportionally less DP overhead, so batch 384 should cut convnext extraction to roughly a third.
# "img" bumped 224 -> 256: a 224-trained ConvNeXt used as a frozen extractor typically yields BETTER
# features at slightly higher resolution (the train/test-resolution "FixRes" effect), a cheap quality
# gain now that the batch bump reclaimed the time. (For a bigger jump use the convnext_small.fb_in22k_
# ft_in1k_384 checkpoint at img=384 — but that ~triples its extraction time, so it's a polish-run opt-in.)
EXTRA_BACKBONES = [
    {"name": "convnext_small", "kind": "conv", "timm_id": "convnext_small.fb_in22k_ft_in1k", "img": 256, "batch": 384},
    {"name": "siglip2_l", "kind": "token", "timm_id": "vit_large_patch16_siglip_256.v2_webli", "img": 256, "batch": 256},
]
# how to combine per-backbone OOF probs into one ensemble (fix D, generalized to N backbones):
#   "weight_tune"   - extend the existing per-class coordinate-ascent weight search across backbones too
#   "logistic_meta" - stack OOF probs from all backbones as features, fit multinomial LogisticRegression
#   "auto"          - fit both on the tune-half of OOF, keep whichever scores higher on the held-out
#                     report-half (still 100% train/OOF-derived -> no Data Uji involved, compliant)
ENSEMBLE_METHOD = "auto"
# a backbone listed here must EARN its place: honestly compared (K-fold, best-of-{weight_tune,
# logistic_meta}) against the ensemble WITHOUT it. If it doesn't improve the held-out score it's dropped
# entirely (a true weight of 0), so adding it is zero-risk -- a useless SigLIP collapses the result back
# to the exact dinov2+convnext baseline instead of quietly dragging it down.
PROBATION_BACKBONES = ["siglip2_l"]
# --- kNN retrieval / label propagation (fix N): exploits this dataset's near-dup-heavy structure
# (110/1458 test imgs are exact train dups + more near-dups) that the frozen features capture perfectly.
# Similarity-GATED + OOF-tuned, so it is a strict no-op on novel/low-sim rows and whenever the tuned
# alpha ends up 0 -> zero-regret, a smooth generalization of the existing cos>=0.98 exact-dup override. ---
RUN_KNN_RETRIEVAL = True   # master switch; whole feature is a strict no-op if False or if tuned alpha==0
KNN_K = 20                 # neighbours per query in the frozen dinov2 feature space
KNN_POWER = 8.0            # neighbour weight = clip(cos,0,None)**power (high => near-dups sim~1 dominate)
KNN_SIM_LO = 0.85          # below this top-sim: gate=0, model prediction untouched
KNN_SIM_HI = 0.95          # at/above this top-sim: gate=1, full (alpha-capped) retrieval weight
# backbone id string used as part of the feature-cache key; folds in the fine-tune config so a cache
# entry from a frozen run is never mistakenly reused for a partially-unfrozen run (or vice versa).
DINOV2_BACKBONE_ID = DINOV2_BACKBONE + (
    f"_ft{UNFREEZE_LAST_N_BLOCKS}e{BACKBONE_FT_EPOCHS}lr{BACKBONE_LR}" if UNFREEZE_LAST_N_BLOCKS > 0 else "")

# --- on-disk feature cache -----------------------------------------------------------------------
# Every backbone above is FROZEN on the default path (never fine-tuned unless UNFREEZE_LAST_N_BLOCKS>0),
# so extract_features()/extract_features_multiscale() are pure deterministic functions of
# (backbone+finetune-state, img size, preprocess mode, pooling mode, which exact images, flip/scale TTA
# view). Re-running the notebook (e.g. iterating on head HPO / ensemble method) otherwise redoes the
# most expensive step -- a full GPU forward pass over ~26.5k train + ~1.5k test images per backbone --
# from scratch every time. Caching to disk skips that whenever the relevant config hasn't changed.
USE_FEATURE_CACHE = True
FEAT_CACHE_DIR = f"{os.environ.get('KAGGLE_WORKING', '/kaggle/working')}/feat_cache"

# --- external suspicious/noise-label list (e.g. a prior suspect_labels.csv, manually reviewed) -----
# Dropped BEFORE dedup/fold assignment so noisy rows never enter a CV fold and features are never
# extracted for them (the drop also naturally busts the feature cache above, since the cache key
# includes a hash of the exact path list).
DROP_NOISE_LABELS = True
NOISE_LABELS_DIR = "/kaggle/input/datasets/ivanwllm/noiselabels4"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=1.0.0"], check=False)

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import timm
from timm.data import create_transform, resolve_data_config, Mixup
from timm.loss import SoftTargetCrossEntropy, LabelSmoothingCrossEntropy
from PIL import Image
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)
def seed_worker(_):
    ws = torch.initial_seed() % 2**32; np.random.seed(ws); random.seed(ws)
seed_all(); torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()
T0 = time.time()
def lap(tag): print(f"[{time.time()-T0:7.1f}s] {tag}", flush=True)
print("device:", DEVICE, "| n_gpu:", N_GPU,
      "|", ", ".join(torch.cuda.get_device_name(i) for i in range(N_GPU)) if DEVICE == "cuda" else "")
try:
    BICUBIC = T.InterpolationMode.BICUBIC
except Exception:
    BICUBIC = Image.BICUBIC

# note whether backbone WEIGHT downloads (not features) are already cached from a Kaggle dataset mount.
if os.environ.get("HF_HUB_CACHE"):
    print(f"[cache] HF_HUB_CACHE={os.environ['HF_HUB_CACHE']} -> backbone weight downloads will be "
          f"skipped if already present there (set this to a Kaggle dataset dir to avoid re-download).")
os.makedirs(FEAT_CACHE_DIR, exist_ok=True)

# --- restore feature cache across separate Kaggle "Save & Run All" commits ---------------------
# /kaggle/working (where FEAT_CACHE_DIR lives) is wiped clean at the start of EVERY separate commit,
# so the cache above only carries over automatically within one interactive session. To also skip
# re-extraction on the NEXT commit: after a run finishes, turn its /kaggle/working/feat_cache output
# into a Kaggle Dataset (Output tab -> "New Dataset"), then attach that dataset as input on the next
# run -- no code/config change needed, this scans every /kaggle/input/*/feat_cache and copies any
# cached .npy files not already present into the (writable) working cache dir before extraction.
for _prev_dir in glob.glob("/kaggle/input/*/feat_cache") + glob.glob("/kaggle/input/*/*/feat_cache"):
    _n_restored = 0
    for _f in glob.glob(os.path.join(_prev_dir, "*.npy")):
        _dst = os.path.join(FEAT_CACHE_DIR, os.path.basename(_f))
        if not os.path.exists(_dst):
            shutil.copy2(_f, _dst); _n_restored += 1
    if _n_restored:
        print(f"[cache] restored {_n_restored} cached feature file(s) from previous-run dataset: {_prev_dir}")


## 2. Dataset Loading

Reconstructs the dataset from the Kaggle input shards, builds the labeled `df` (path, label), audits/drops
corrupt images, applies the user-supplied noise-label relabeling, computes the perceptual-hash (DCT pHash)
near-duplicate groups, and assigns dedup-aware `StratifiedGroupKFold` CV folds. **Verbatim from the original
single cell** — the fold assignment must stay identical to keep OOF/CV honest and reproducible.


In [ ]:
# ============================================================
# 1. RECONSTRUCT DATA
# ============================================================
WORK = "/kaggle/working"; DATA = f"{WORK}/BDC2026"
if not os.path.exists(f"{DATA}/train"):
    parts = sorted(glob.glob("/kaggle/input/**/data.zip.*", recursive=True))
    print("found", len(parts), "parts")
    if not parts:
        print("DEBUG /kaggle/input:", os.listdir("/kaggle/input"))
        for d in os.listdir("/kaggle/input"):
            print("  ", d, "->", os.listdir(os.path.join("/kaggle/input", d))[:6])
        raise SystemExit("No data parts found under /kaggle/input")
    print("reconstructing zip from", len(parts), "parts...")
    with open(f"{WORK}/data.zip", "wb") as o:
        for p in parts:
            with open(p, "rb") as f: shutil.copyfileobj(f, o, 1 << 24)
    with zipfile.ZipFile(f"{WORK}/data.zip") as z: z.extractall(WORK)
    os.remove(f"{WORK}/data.zip")
TRAIN, TEST, SUBT = f"{DATA}/train", f"{DATA}/test", f"{DATA}/submission.csv"
print("train:", os.path.exists(TRAIN), "| test:", os.path.exists(TEST), "| submission tmpl:", os.path.exists(SUBT))


In [ ]:
# ============================================================
# 2. DATAFRAME + CORRUPT AUDIT
# ============================================================
def build_df():
    m = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
    rows = []
    for folder, lab in m.items():
        d = os.path.join(TRAIN, folder)
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                rows.append((os.path.join(d, fn), lab))
    return pd.DataFrame(rows, columns=["path", "label"]).sort_values("path").reset_index(drop=True)

df = build_df()
lap(f"df built: {len(df)} images")

def audit_corrupt(paths):
    bad = []
    for p in paths:
        try:
            with Image.open(p) as im: im.verify()
        except Exception:
            bad.append(p)
    return bad

# corrupt audit (and the noise-label drop below) always run regardless of RUN_EDA: both change which
# rows are in `df`, which feeds the dedup/CV split -> not just informational, so never toggled off.
bad_files = audit_corrupt(df["path"].tolist())
print(f"[EDA] corrupt/unreadable files: {len(bad_files)} / {len(df)}")
if bad_files:
    print("[EDA] sample corrupt paths:", bad_files[:5])
    df = df[~df["path"].isin(bad_files)].reset_index(drop=True)
    print(f"[EDA] dropped corrupt rows -> remaining {len(df)}")
lap("corrupt audit done")
# test set is audited too (fix I): previously corrupt test files silently fell back to a blank
# black image in ImgOnlyDS with no visibility into how many were affected.
if os.path.exists(TEST):
    test_paths_preview = glob.glob(os.path.join(TEST, "*"))
    bad_test = audit_corrupt(test_paths_preview)
    print(f"[EDA] TEST corrupt/unreadable files: {len(bad_test)} / {len(test_paths_preview)} "
          f"(these will fall back to a blank image at inference; template row order is preserved)")
    if bad_test:
        print("[EDA] sample corrupt TEST paths:", bad_test[:5])


In [ ]:
# ============================================================
# 2a. DROP USER-SUPPLIED SUSPICIOUS/NOISE LABELS (manual review, e.g. a prior suspect_labels.csv)
# ============================================================
if DROP_NOISE_LABELS:
    noise_csvs = sorted(glob.glob(os.path.join(NOISE_LABELS_DIR, "*.csv"))) if os.path.exists(NOISE_LABELS_DIR) else []
    if not noise_csvs:
        print(f"[NOISE] DROP_NOISE_LABELS=True but no CSV found under {NOISE_LABELS_DIR} -- skipping.")
    else:
        noise_df = pd.concat([pd.read_csv(p) for p in noise_csvs], ignore_index=True)
        RELABEL_CONF_TH = 0.999  # atau 0.999
        noise_df = noise_df[noise_df["oof_conf"] >= RELABEL_CONF_TH]
        # accept whichever column identifies the image, by full path or bare filename
        path_col = next((c for c in ("path", "filepath", "file", "filename", "image", "id") if c in noise_df.columns), None)
        if "oof_pred" not in noise_df.columns:
            raise ValueError("CSV must contain column 'oof_pred'")
        if path_col is None:
            print(f"[NOISE][WARN] no recognizable path/filename column in {noise_csvs} "
                  f"(cols={list(noise_df.columns)}) -- skipping.")
        else:
            relabel_map = {
                os.path.basename(str(row[path_col])): int(row["oof_pred"])
                for _, row in noise_df.iterrows()
            } 
            mask = df["path"].apply(lambda p: os.path.basename(p) in relabel_map)
            before = mask.sum()
            df.loc[mask, "label"] = (df.loc[mask, "path"].apply(lambda p: relabel_map[os.path.basename(p)]).astype(int))
            print(f"[NOISE] loaded {len(relabel_map)} suspicious labels from {len(noise_csvs)} csv(s) "f"under {NOISE_LABELS_DIR} (col='{path_col}') -> relabeled {before} train rows before CV split")
            lap("noise-relabel done")


In [ ]:
# ============================================================
# 2b. FULL-DATASET aHash NEAR-DUP DEDUP (LSH-blocked, union-find)  -> honest CV only
# ============================================================
_DCT_BASIS_CACHE = {}
def _dct_basis(n):
    """Orthonormal DCT-II basis matrix [n,n] (so a 2D DCT is B @ img @ B.T). Cached per size."""
    if n not in _DCT_BASIS_CACHE:
        k = np.arange(n)
        basis = np.cos(np.pi * (2 * k[None, :] + 1) * k[:, None] / (2 * n)).astype(np.float32)
        basis[0, :] *= 1.0 / np.sqrt(2.0)
        _DCT_BASIS_CACHE[n] = basis * np.sqrt(2.0 / n)
    return _DCT_BASIS_CACHE[n]

def compute_phash(path, size=DEDUP_PHASH_SIZE, factor=DEDUP_DCT_FACTOR):
    """Perceptual DCT pHash (replaces the old 8x8 average-hash, which false-matched any two images with
    a similar brightness layout -> chaining). Resize to (size*factor)px grayscale, take the 2D DCT, keep
    the top-left size x size block of LOW-frequency coefficients (perceptual structure, robust to small
    lighting/scale changes), threshold against their median (excluding the DC term) -> a size*size-bit hash."""
    img_px = size * factor
    with Image.open(path) as im:
        small = np.asarray(im.convert("L").resize((img_px, img_px), Image.BILINEAR), dtype=np.float32)
    B = _dct_basis(img_px)
    dct = B @ small @ B.T
    low = dct[:size, :size]                 # top-left low-frequency block
    thr = np.median(low.flatten()[1:])      # median excluding DC (index 0) -> stable 50/50 bit split
    bits = low > thr
    h = 0
    for b in bits.flatten():
        h = (h << 1) | int(b)
    return h

def hamming(a, b): return bin(a ^ b).count("1")

def build_lsh_buckets(hashes, n_blocks=DEDUP_LSH_BLOCKS, total_bits=DEDUP_PHASH_SIZE * DEDUP_PHASH_SIZE):
    bits_per_block = total_bits // n_blocks
    buckets = [dict() for _ in range(n_blocks)]
    for idx, h in enumerate(hashes):
        for b in range(n_blocks):
            key = (h >> (b * bits_per_block)) & ((1 << bits_per_block) - 1)
            buckets[b].setdefault(key, []).append(idx)
    return buckets

def find_near_dup_pairs(hashes, max_dist=DEDUP_MAX_HAMMING):
    buckets = build_lsh_buckets(hashes)
    candidates = set()
    for buck in buckets:
        for idxs in buck.values():
            if len(idxs) > 1:
                for i in range(len(idxs)):
                    for j in range(i + 1, len(idxs)):
                        candidates.add((idxs[i], idxs[j]))
    return [(i, j) for (i, j) in candidates if hamming(hashes[i], hashes[j]) <= max_dist]

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.parent[ra] = rb

if RUN_DEDUP:
    lap("computing full-dataset DCT pHash (train)")
    train_hashes = [compute_phash(p) for p in df["path"].tolist()]
    dup_pairs_full = find_near_dup_pairs(train_hashes)
    uf = UnionFind(len(df))
    for i, j in dup_pairs_full: uf.union(i, j)
    df["dup_group"] = [uf.find(i) for i in range(len(df))]

    # GUARD (new): dissolve any oversized cluster back into singletons. A dup-cluster larger than a small
    # fraction of the dataset is a hash artifact (false-positive chaining), not real duplicates, and if
    # left intact StratifiedGroupKFold is forced to pile it all into ONE fold -> the degenerate fold
    # sizes we saw last run. Splitting these back into singletons only risks re-separating IMAGES THAT
    # WERE NEVER REALLY DUPLICATES, so it's safe, and it guarantees balanced folds.
    cap = max(2, int(DEDUP_MAX_GROUP_FRAC * len(df)))
    grp_sizes = df["dup_group"].value_counts()
    oversized = grp_sizes[grp_sizes > cap]
    if len(oversized):
        bad = set(oversized.index.tolist())
        new_group = df["dup_group"].values.copy()
        nxt = int(new_group.max()) + 1
        for i in range(len(new_group)):
            if new_group[i] in bad:
                new_group[i] = nxt; nxt += 1
        df["dup_group"] = new_group
        print(f"[DEDUP][GUARD] dissolved {len(oversized)} oversized cluster(s) > {cap} imgs "
              f"(largest were {oversized.head(3).tolist()}) into singletons -> prevents degenerate folds")

    n_clusters = df["dup_group"].nunique()
    sizes_after = df["dup_group"].value_counts()
    n_in_dup_cluster = int((sizes_after[sizes_after > 1]).sum())
    print(f"[DEDUP] near-dup pairs (Hamming<={DEDUP_MAX_HAMMING}, DCT pHash): {len(dup_pairs_full)} | "
          f"{len(df)} imgs -> {n_clusters} groups | {n_in_dup_cluster} imgs in a cluster of size>1 | "
          f"largest cluster now = {int(sizes_after.max())} imgs")
    lap("dedup clustering done")
else:
    df["dup_group"] = np.arange(len(df))

df["fold"] = -1
if RUN_DEDUP:
    sgkf = StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED)
    for f, (_, vi) in enumerate(sgkf.split(df, df.label, groups=df["dup_group"])):
        df.loc[df.index[vi], "fold"] = f
    # audit: fold balance + residual leakage. Reuse dup_pairs_full (already computed) instead of
    # recomputing ~1.1M loose pairs just to print a number, as the old H<=8 audit did (~30s wasted).
    print("[DEDUP] fold sizes:", df.groupby("fold").size().tolist())
    cross_fold = sum(df["fold"].iloc[i] != df["fold"].iloc[j] for i, j in dup_pairs_full)
    print(f"[DEDUP] near-dup pairs crossing a fold boundary: {cross_fold} / {len(dup_pairs_full)} "
          f"(pairs inside dissolved oversized clusters may now cross -> expected, they weren't real dups)")
else:
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
    for f, (_, vi) in enumerate(skf.split(df, df.label)): df.loc[vi, "fold"] = f
lap("folds assigned")


### 2.1 Configuration & dataset summary (read-only)

Human-readable recap of the cleaned dataset and the active configuration. Does not affect `df`, folds, or
any downstream cell.


In [ ]:
# read-only summary — safe to skip, does not affect the pipeline
print(f"Total images after cleaning : {len(df)}")
print(f"Classes                     : {CLASS_NAMES}")
print(f"Class counts                : {df['label'].value_counts().sort_index().to_dict()}")
print(f"N folds                     : {N_FOLDS}  (StratifiedGroupKFold on dup_group)" if RUN_DEDUP else f"N folds : {N_FOLDS} (StratifiedKFold)")
print(f"Fold sizes                  : {df.groupby('fold').size().to_dict()}")
print(f"MODE                        : {MODE}")
print(f"DINOv2 backbone             : {DINOV2_BACKBONE if MODE=='dinov2_linear' else 'n/a'}")
print(f"IMG size / preprocess       : {IMG} / {PREPROCESS}")
df.head()


## 3. Exploratory Data Analysis (NEW)

Everything in this section is **new, read-only analysis**. The first cell below is the original `run_eda(df)`
(kept verbatim — it prints the text summary and returns `eda_result`); every cell after it is additive and
only *reads* `df` / image files. Sampling uses `random_state=SEED` so nothing here perturbs the training RNG.


In [ ]:
# ============================================================
# 3. EDA BLOCK (sampled per class; ends with an explicit preprocessing recommendation)
# ============================================================
def run_eda(df, sample_per_class=EDA_SAMPLE_PER_CLASS, seed=SEED):
    """Pure analysis pass over df (class balance, resolution/aspect ratio, color mode, per-class RGB
    stats, blur-proxy, pad-vs-crop recommendation). Returns a dict instead of only printing, so callers
    can reuse the numbers. Does NOT touch `df`, dedup, or CV split -- those must run unconditionally
    (see section 2/2a/2b above), this function is purely informational and safe to skip entirely."""
    print("\n===== EDA =====")
    counts = df.label.value_counts().sort_index()
    ratio = counts.max() / counts.min()
    print("[EDA-1] class counts:", {CLASS_NAMES[k]: int(v) for k, v in counts.items()},
          f"| max/min imbalance ratio = {ratio:.2f}")

    samp_parts = []
    for lbl, g in df.groupby("label"):
        samp_parts.append(g.sample(min(len(g), sample_per_class), random_state=seed))
    samp = pd.concat(samp_parts)

    res_w, res_h, ars, means_rgb, stds_rgb, blur_scores, modes = [], [], [], [], [], [], []
    for _, row in samp.iterrows():
        try:
            with Image.open(row.path) as im:
                modes.append(im.mode)
                im_rgb = im.convert("RGB"); w, h = im_rgb.size
                res_w.append(w); res_h.append(h); ars.append(w / h)
                arr = np.asarray(im_rgb.resize((64, 64)), dtype=np.float32)
                means_rgb.append(arr.reshape(-1, 3).mean(0))
                stds_rgb.append(arr.reshape(-1, 3).std(0))
                gray = arr.mean(2); gy, gx = np.gradient(gray)
                blur_scores.append((gx**2 + gy**2).var())
        except Exception:
            continue
    res_w, res_h, ars = np.array(res_w), np.array(res_h), np.array(ars)
    means_rgb, stds_rgb, blur_scores = np.array(means_rgb), np.array(stds_rgb), np.array(blur_scores)

    ar_std = float(ars.std())
    print(f"[EDA-2] resolution WxH: mean=({res_w.mean():.0f}x{res_h.mean():.0f}) "
          f"min=({res_w.min()}x{res_h.min()}) max=({res_w.max()}x{res_h.max()}) "
          f"aspect_ratio mean={ars.mean():.2f} std={ar_std:.2f}")
    from collections import Counter
    color_modes = dict(Counter(modes))
    print("[EDA-3] color-mode distribution (raw):", color_modes)

    samp_labels = samp.label.values[:len(means_rgb)]
    per_class_stats = {}
    for c in range(NUM_CLASSES):
        sel = samp_labels == c
        if sel.sum() > 0:
            cname = CLASS_NAMES[c]
            per_class_stats[cname] = {
                "mean_rgb": means_rgb[sel].mean(0).round(1).tolist(),
                "contrast": float(stds_rgb[sel].mean(0).mean()),
                "brightness": float(means_rgb[sel].mean()),
                "blur_proxy": float(blur_scores[sel].mean()),
            }
            print(f"[EDA-6] class={cname:<11} mean RGB={per_class_stats[cname]['mean_rgb']} "
                  f"contrast(std)={per_class_stats[cname]['contrast']:.1f} "
                  f"brightness={per_class_stats[cname]['brightness']:.1f} blur-proxy={per_class_stats[cname]['blur_proxy']:.1f}")
    if RUN_DEDUP:
        print(f"[EDA-5] near-dup pairs (Hamming<={DEDUP_MAX_HAMMING}): {len(dup_pairs_full)} across "
              f"{n_clusters} groups -> StratifiedGroupKFold used (no near-dup crosses train/val in a fold).")

    # ---- EDA -> preprocessing recommendation (informational only; PREPROCESS is still set manually
    # in config regardless of RUN_EDA, so nothing downstream depends on this being computed) ----
    rec = "pad-to-square" if ar_std > 0.15 else "center-crop"
    print(f"[EDA->PREP] aspect_ratio std={ar_std:.2f} -> recommend {rec}. "
          f"Currently PREPROCESS='{PREPROCESS}', IMG={IMG}. "
          f"(High aspect std => center-crop clips the long edge, so 'pad' preserves the whole object.)")
    print("===== END EDA =====\n")

    return {
        "class_counts": {CLASS_NAMES[k]: int(v) for k, v in counts.items()},
        "imbalance_ratio": float(ratio),
        "resolution": {"mean_w": float(res_w.mean()), "mean_h": float(res_h.mean()),
                        "min_w": int(res_w.min()), "min_h": int(res_h.min()),
                        "max_w": int(res_w.max()), "max_h": int(res_h.max())},
        "aspect_ratio": {"mean": float(ars.mean()), "std": ar_std},
        "color_modes": color_modes,
        "per_class": per_class_stats,
        "preprocess_recommendation": rec,
    }

eda_result = run_eda(df) if RUN_EDA else None
lap("EDA done" if RUN_EDA else "EDA skipped (RUN_EDA=False)")


In [ ]:
# visualize the original run_eda() per-class summary (eda_result), which is text-only above
import matplotlib.pyplot as plt

if eda_result is not None:
    _pc = pd.DataFrame(eda_result["per_class"]).T
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].bar(_pc.index, _pc["brightness"], color="#4C72B0"); axes[0].set_title("Brightness (run_eda)")
    axes[1].bar(_pc.index, _pc["contrast"], color="#DD8452"); axes[1].set_title("Contrast/std (run_eda)")
    axes[2].bar(_pc.index, _pc["blur_proxy"], color="#55A868"); axes[2].set_title("Blur proxy (run_eda)")
    for ax in axes: ax.tick_params(axis="x", rotation=15)
    plt.tight_layout(); plt.savefig(f"{WORK}/eda_run_eda_summary.png", dpi=120); plt.show()
else:
    print("[note] RUN_EDA=False -> eda_result is None, nothing to plot here.")


### 3.1 File Integrity & Inventory

In [ ]:
# read-only inventory recap
import matplotlib.pyplot as plt

fnames = df["path"].apply(lambda p: os.path.basename(p))
dupe_names = fnames[fnames.duplicated(keep=False)]
n_dupe_rows = int(len(dupe_names))
print(f"Total usable images        : {len(df)}")
print(f"Total classes               : {df['label'].nunique()}  -> {CLASS_NAMES}")
print(f"Corrupt/unreadable dropped  : {len(bad_files)}  (see section 2)")
print(f"Duplicate-filename rows     : {n_dupe_rows}  ({dupe_names.nunique()} distinct name(s))")
if n_dupe_rows:
    display(df.loc[dupe_names.index, ["path", "label"]].sort_values("path").head(20))

inventory = pd.Series({
    "usable images": len(df),
    "corrupt (dropped)": len(bad_files),
    "duplicate filenames": n_dupe_rows,
})
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(inventory.index, inventory.values, color=["#4C72B0", "#C44E52", "#DD8452"])
axes[0].set_title("Dataset inventory"); axes[0].tick_params(axis="x", rotation=15)
class_counts_plot = df["label"].map(CLASS_NAMES).value_counts()
axes[1].bar(class_counts_plot.index, class_counts_plot.values, color="#55A868")
axes[1].set_title("Usable images per class")
plt.tight_layout(); plt.savefig(f"{WORK}/eda_inventory.png", dpi=120); plt.show()


### 3.2 Per-image Metadata Extraction

In [ ]:
import colorsys
from PIL import Image as _Image

def extract_image_metadata(paths, labels):
    rows = []
    for p, lab in zip(paths, labels):
        try:
            with _Image.open(p) as im:
                w, h = im.size
                filesize = os.path.getsize(p)
                bpp = (filesize * 8) / (w * h) if w * h else np.nan
                rgb = np.asarray(im.convert("RGB").resize((128, 128)), dtype=np.float32)
                hsv = np.asarray(im.convert("HSV").resize((128, 128)), dtype=np.float32)
                gray = rgb.mean(axis=2)
                rows.append({
                    "path": p, "label": lab, "width": w, "height": h,
                    "aspect_ratio": w / h, "megapixels": (w * h) / 1e6,
                    "filesize_kb": filesize / 1024, "bpp": bpp,
                    "r_mean": rgb[..., 0].mean(), "g_mean": rgb[..., 1].mean(), "b_mean": rgb[..., 2].mean(),
                    "r_std": rgb[..., 0].std(), "g_std": rgb[..., 1].std(), "b_std": rgb[..., 2].std(),
                    "h_mean": hsv[..., 0].mean(), "s_mean": hsv[..., 1].mean(), "v_mean": hsv[..., 2].mean(),
                    "gray_mean": gray.mean(), "gray_std": gray.std(),
                })
        except Exception:
            continue
    return pd.DataFrame(rows)

_meta_samp = pd.concat([g.sample(min(len(g), EDA_SAMPLE_PER_CLASS), random_state=SEED)
                        for _, g in df.groupby("label")])
meta_df = extract_image_metadata(_meta_samp["path"].tolist(), _meta_samp["label"].tolist())
meta_df["class_name"] = meta_df["label"].map(CLASS_NAMES)
print(f"metadata extracted for {len(meta_df)} / {len(_meta_samp)} sampled images")
meta_df.describe()


In [ ]:
# quick visual overview of every extracted metadata field (one histogram grid)
_meta_numeric = ["width", "height", "aspect_ratio", "megapixels", "filesize_kb", "bpp",
                  "r_mean", "g_mean", "b_mean", "h_mean", "s_mean", "v_mean", "gray_mean", "gray_std"]
fig, axes = plt.subplots(4, 4, figsize=(14, 11))
for ax, col in zip(axes.ravel(), _meta_numeric):
    ax.hist(meta_df[col], bins=25, color="#4C72B0")
    ax.set_title(col, fontsize=9)
for ax in axes.ravel()[len(_meta_numeric):]:
    ax.axis("off")
fig.suptitle("Per-image metadata: distribution overview")
plt.tight_layout(); plt.savefig(f"{WORK}/eda_metadata_overview.png", dpi=120); plt.show()


### 3.3 Class Distribution & Imbalance Ratio

In [ ]:
import matplotlib.pyplot as plt

counts = df["label"].value_counts().sort_index()
pct = (counts / counts.sum() * 100).round(2)
imbalance_table = pd.DataFrame({
    "class": [CLASS_NAMES[k] for k in counts.index],
    "count": counts.values,
    "pct": pct.values,
    "imbalance_ratio_vs_min": (counts / counts.min()).round(2).values,
})
display(imbalance_table)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar([CLASS_NAMES[k] for k in counts.index], counts.values, color=["#4C72B0", "#DD8452", "#55A868"])
ax[0].set_title("Class count"); ax[0].set_ylabel("images")
ax[1].pie(counts.values, labels=[CLASS_NAMES[k] for k in counts.index], autopct="%1.1f%%")
ax[1].set_title("Class share")
plt.tight_layout(); plt.savefig(f"{WORK}/eda_class_distribution.png", dpi=120); plt.show()
print(f"max/min imbalance ratio = {counts.max()/counts.min():.2f}")


### 3.4 Fold Composition (NEW)

In [ ]:
# 3.4 Fold composition: class balance across the StratifiedGroupKFold folds (read-only)
import matplotlib.pyplot as plt
fold_tab = df.pivot_table(index="fold", columns="label", values="path", aggfunc="count", fill_value=0)
fold_tab.columns = [CLASS_NAMES[c] for c in fold_tab.columns]
display(fold_tab)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fold_tab.plot(kind="bar", stacked=True, ax=axes[0], color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("Images per fold (stacked by class)"); axes[0].set_xlabel("fold"); axes[0].tick_params(axis="x", rotation=0)
fold_tab.div(fold_tab.sum(1), axis=0).plot(kind="bar", stacked=True, ax=axes[1],
                                           color=["#4C72B0", "#DD8452", "#55A868"], legend=False)
axes[1].set_title("Class proportion per fold (should be ~constant)"); axes[1].set_xlabel("fold"); axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.savefig(f"{WORK}/eda_fold_composition.png", dpi=120); plt.show()


### 3.5 Sample Gallery

In [ ]:
def show_gallery(paths_labels, ncols=6, title=""):
    n = len(paths_labels)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.2*ncols, 2.4*nrows))
    axes = np.array(axes).reshape(-1)
    for ax, (p, lab) in zip(axes, paths_labels):
        try:
            with _Image.open(p) as im:
                ax.imshow(im.convert("RGB"))
        except Exception:
            pass
        ax.set_title(lab, fontsize=8); ax.axis("off")
    for ax in axes[len(paths_labels):]:
        ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

for c in range(NUM_CLASSES):
    samp = df[df.label == c].sample(min(6, (df.label == c).sum()), random_state=SEED)
    show_gallery(list(zip(samp["path"], [CLASS_NAMES[c]] * len(samp))), title=f"Sample gallery: {CLASS_NAMES[c]}")

# edge cases: extreme aspect ratio (proxy for "hard to pad/crop cleanly")
if len(meta_df):
    edge = meta_df.reindex(meta_df["aspect_ratio"].sub(1).abs().sort_values(ascending=False).index).head(6)
    show_gallery(list(zip(edge["path"], edge["class_name"] + " ar=" + edge["aspect_ratio"].round(2).astype(str))),
                 title="Edge cases: most extreme aspect ratio")

# difficult samples from OOF (only if section 6/7 already ran in this session)
if "oof" in globals() and "filled" in globals() and filled.any():
    _pred = oof.argmax(1); _y = df.label.values
    _wrong = df.index[filled & (_pred != _y)]
    if len(_wrong):
        _conf = oof.max(1)
        _hardest = df.loc[_wrong].assign(conf=_conf[_wrong], pred=_pred[_wrong]).sort_values("conf", ascending=False).head(6)
        show_gallery(list(zip(_hardest["path"],
                              _hardest["label"].map(CLASS_NAMES) + "->" + _hardest["pred"].map(CLASS_NAMES))),
                     title="Difficult samples: high-confidence OOF errors")
else:
    print("[placeholder] OOF-based difficult-sample highlighting needs `oof`/`filled` from section 6 "
          "to be in memory — run section 4-6 first, then re-run this cell.")


### 3.6 Resolution & Aspect Ratio

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0,0].hist(meta_df["width"], bins=40, color="#4C72B0"); axes[0,0].set_title("Width histogram")
axes[0,1].hist(meta_df["height"], bins=40, color="#DD8452"); axes[0,1].set_title("Height histogram")
axes[1,0].hist(meta_df["aspect_ratio"], bins=40, color="#55A868"); axes[1,0].set_title("Aspect ratio histogram")
sc = axes[1,1].scatter(meta_df["width"], meta_df["height"], c=meta_df["label"], cmap="viridis", s=8, alpha=0.6)
axes[1,1].set_xlabel("width"); axes[1,1].set_ylabel("height"); axes[1,1].set_title("Width vs height (colored by class)")
plt.tight_layout(); plt.savefig(f"{WORK}/eda_resolution.png", dpi=120); plt.show()

print(f"Resolution summary: mean=({meta_df.width.mean():.0f}x{meta_df.height.mean():.0f}) "
      f"min=({meta_df.width.min()}x{meta_df.height.min()}) max=({meta_df.width.max()}x{meta_df.height.max()})")
print(f"Aspect ratio: mean={meta_df.aspect_ratio.mean():.2f} std={meta_df.aspect_ratio.std():.2f} "
      f"(PREPROCESS='{PREPROCESS}', IMG={IMG})")


### 3.7 File Size & Compression (per class)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for c in range(NUM_CLASSES):
    sub = meta_df[meta_df.label == c]
    axes[0].hist(sub["filesize_kb"], bins=30, alpha=0.5, label=CLASS_NAMES[c])
    axes[1].hist(sub["bpp"], bins=30, alpha=0.5, label=CLASS_NAMES[c])
axes[0].set_title("Filesize (KB) per class"); axes[0].legend()
axes[1].set_title("Bits-per-pixel per class"); axes[1].legend()
plt.tight_layout(); plt.savefig(f"{WORK}/eda_filesize_bpp.png", dpi=120); plt.show()

display(meta_df.groupby("class_name")[["filesize_kb", "bpp"]].agg(["mean", "std", "median"]))


### 3.8 Photometric Statistics (per class)

In [ ]:
meta_df["brightness"] = meta_df[["r_mean","g_mean","b_mean"]].mean(axis=1)
meta_df["contrast"] = meta_df[["r_std","g_std","b_std"]].mean(axis=1)
meta_df["luma"] = 0.299*meta_df["r_mean"] + 0.587*meta_df["g_mean"] + 0.114*meta_df["b_mean"]

photo_cols = ["r_mean","g_mean","b_mean","h_mean","s_mean","v_mean","brightness","contrast","luma"]
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, col in zip(axes.ravel(), photo_cols):
    meta_df.boxplot(column=col, by="class_name", ax=ax)
    ax.set_title(col); ax.set_xlabel("")
plt.suptitle("")
plt.tight_layout(); plt.savefig(f"{WORK}/eda_photometric.png", dpi=120); plt.show()

display(meta_df.groupby("class_name")[photo_cols].mean().round(1))
print(f"Current ColorJitter (train-aug): brightness=0.3 contrast=0.3 saturation=0.3 hue=0.05 "
      f"(see build_train_aug_transform in section 4/5A) -- compare against the spread above.")


### 3.9 Sharpness / Blur (Laplacian variance)

In [ ]:
# simple, dependency-free blur proxy: variance of the image gradient (same idea as a Laplacian,
# no cv2 needed -- reuses the grayscale array already computed per image).
def blur_proxy(path):
    with _Image.open(path) as im:
        gray = np.asarray(im.convert("L").resize((128, 128)), dtype=np.float32)
    gy, gx = np.gradient(gray)
    return float((gx**2 + gy**2).var())

meta_df["blur_score"] = meta_df["path"].apply(blur_proxy)

fig, ax = plt.subplots(figsize=(7, 4))
for c in range(NUM_CLASSES):
    ax.hist(meta_df.loc[meta_df.label == c, "blur_score"], bins=30, alpha=0.5, label=CLASS_NAMES[c])
ax.set_title("Sharpness / blur-proxy distribution per class (higher = sharper)")
ax.set_xlabel("gradient-variance blur score"); ax.legend()
plt.tight_layout(); plt.savefig(f"{WORK}/eda_blur.png", dpi=120); plt.show()


### 3.10 Per-class Mean Image

In [ ]:
def mean_image(paths, size=128, n_max=300, seed=SEED):
    rng = np.random.default_rng(seed)
    sel = rng.choice(paths, size=min(n_max, len(paths)), replace=False)
    acc = np.zeros((size, size, 3), dtype=np.float64); n = 0
    for p in sel:
        try:
            with _Image.open(p) as im:
                w, h = im.convert("RGB").size
                s = max(w, h)
                im2 = im.convert("RGB")
                pad = _Image.new("RGB", (s, s), (128,128,128))
                pad.paste(im2, ((s-w)//2, (s-h)//2))
                arr = np.asarray(pad.resize((size, size)), dtype=np.float64)
                acc += arr; n += 1
        except Exception:
            continue
    return (acc / max(n, 1)).astype(np.uint8)

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(4*NUM_CLASSES, 4))
for c in range(NUM_CLASSES):
    m = mean_image(df.loc[df.label == c, "path"].tolist())
    axes[c].imshow(m); axes[c].set_title(f"Mean image: {CLASS_NAMES[c]}"); axes[c].axis("off")
plt.tight_layout(); plt.savefig(f"{WORK}/eda_mean_images.png", dpi=120); plt.show()


### 3.11 Near-duplicate Structure (NEW)

In [ ]:
# 3.11 Near-duplicate structure (read-only): the pHash clusters that drove the group-aware CV split
import matplotlib.pyplot as plt
if RUN_DEDUP and "dup_group" in df.columns:
    sizes = df["dup_group"].value_counts()
    multi = sizes[sizes > 1]
    print(f"Total images                 : {len(df)}")
    print(f"Distinct dup-groups          : {df['dup_group'].nunique()}")
    print(f"Images in a cluster (size>1) : {int(multi.sum())}  across {len(multi)} clusters")
    print(f"Largest cluster              : {int(sizes.max())} images")
    if len(multi):
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.hist(multi.values, bins=range(2, int(multi.max()) + 2), color="#8172B2", align="left")
        ax.set_xlabel("cluster size"); ax.set_ylabel("num clusters")
        ax.set_title("Near-duplicate cluster-size distribution (size>1)")
        plt.tight_layout(); plt.savefig(f"{WORK}/eda_dup_clusters.png", dpi=120); plt.show()
    if "dup_pairs_full" in globals() and len(dup_pairs_full):
        i, j = dup_pairs_full[0]
        show_gallery([(df.iloc[i]["path"], f"{CLASS_NAMES[df.iloc[i]['label']]} (#{i})"),
                      (df.iloc[j]["path"], f"{CLASS_NAMES[df.iloc[j]['label']]} (#{j})")],
                     ncols=2, title="Example near-duplicate pair (DCT pHash)")
else:
    print("[note] RUN_DEDUP=False -> no dup_group clusters to analyze.")


### 3.12 Train vs Test Distribution Shift

In [ ]:
from scipy import stats as _stats

if os.path.exists(TEST):
    test_paths_for_eda = glob.glob(os.path.join(TEST, "*"))
    _rng = np.random.default_rng(SEED)
    test_sample = list(_rng.choice(test_paths_for_eda, size=min(400, len(test_paths_for_eda)), replace=False))
    test_meta = extract_image_metadata(test_sample, [-1]*len(test_sample))

    compare_cols = ["width", "height", "aspect_ratio", "r_mean", "g_mean", "b_mean",
                     "h_mean", "s_mean", "v_mean"]
    ks_rows = []
    for col in compare_cols:
        a, b = meta_df[col].dropna(), test_meta[col].dropna()
        if len(a) and len(b):
            ks = _stats.ks_2samp(a, b)
            ks_rows.append({"feature": col, "ks_stat": round(ks.statistic, 4), "p_value": ks.pvalue})
    ks_table = pd.DataFrame(ks_rows).sort_values("ks_stat", ascending=False)
    display(ks_table)

    fig, axes = plt.subplots(3, 3, figsize=(13, 10))
    for ax, col in zip(axes.ravel(), compare_cols):
        ax.hist(meta_df[col].dropna(), bins=30, alpha=0.5, density=True, label="train")
        ax.hist(test_meta[col].dropna(), bins=30, alpha=0.5, density=True, label="test")
        ax.set_title(col); ax.legend()
    plt.tight_layout(); plt.savefig(f"{WORK}/eda_train_vs_test_shift.png", dpi=120); plt.show()
    print("[EDA-shift] KS stat close to 0 / high p-value => distributions similar; "
          "large KS stat + low p-value => likely distribution shift on that feature.")
else:
    print("[placeholder] TEST directory not found in this environment -- run inside the Kaggle kernel "
          "(after section 2's data reconstruction) to compute the train-vs-test shift comparison.")


## 4. Training (model & loss definitions)

Loss functions, weighted sampler, the DINOv2 frozen-probe path (transforms, masked pooling, dual head, head
HPO, feature cache, per-fold head training, temperature calibration, optional partial unfreeze) and the
optional ConvNeXt fine-tune path. **Verbatim from the original single cell** — these cells only *define*
functions/classes; the actual run happens in section 5.


In [ ]:
# ============================================================
# 4. LOSS + SAMPLER
# ============================================================
class_counts = df.label.value_counts().sort_index().values.astype(np.float64)

def cb_weights(counts, beta=CB_BETA):
    eff_num = 1.0 - np.power(beta, counts); w = (1.0 - beta) / eff_num
    return w / w.sum() * len(counts)

CB_W = torch.tensor(cb_weights(class_counts), dtype=torch.float32, device=DEVICE)
print("[loss] class-balanced weights:", CB_W.cpu().numpy().round(3).tolist())

class CBFocalLoss(nn.Module):
    def __init__(self, weight, gamma=FOCAL_GAMMA):
        super().__init__(); self.weight = weight; self.gamma = gamma
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1); p = logp.exp()
        logp_t = logp.gather(1, target.unsqueeze(1)).squeeze(1)
        p_t = p.gather(1, target.unsqueeze(1)).squeeze(1)
        w_t = self.weight[target]
        return (-w_t * (1 - p_t) ** self.gamma * logp_t).mean()

smooth_ce = LabelSmoothingCrossEntropy(LS)
soft_ce = SoftTargetCrossEntropy()
cb_focal = CBFocalLoss(CB_W)

def sampler(labels, pow=SAMPLER_POW, g=None):
    c = np.bincount(labels, minlength=NUM_CLASSES); w = (c.sum() / np.maximum(c, 1)) ** pow
    return WeightedRandomSampler(torch.as_tensor(w[labels], dtype=torch.double), len(labels), True, generator=g)

def macro_f1(y, p): return f1_score(y, p, average="macro")


In [ ]:
# ============================================================
# 5A. PATH: DINOv2 frozen extractor (CLS+patch, L2-norm, multi-GPU) + dual head (probe + MLP)
# ============================================================
class SquarePad:
    """Pad shorter side to a square using edge-replication (extends plain backgrounds naturally,
    unlike black constant padding which shifts pixel statistics)."""
    def __init__(self, mode="edge"): self.mode = mode
    def __call__(self, im):
        w, h = im.size; s = max(w, h)
        l = (s - w) // 2; r = s - w - l; t = (s - h) // 2; b = s - h - t
        if l == r == t == b == 0: return im
        return TF.pad(im, [l, t, r, b], padding_mode=self.mode)

def build_infer_transform(img, mean, std, mode="pad"):
    ops = []
    if mode == "pad":
        ops += [SquarePad("edge"), T.Resize((img, img), interpolation=BICUBIC)]
    elif mode == "squash":
        ops += [T.Resize((img, img), interpolation=BICUBIC)]
    else:  # "crop": resize shorter side then center-crop (clips long edge)
        ops += [T.Resize(img, interpolation=BICUBIC), T.CenterCrop(img)]
    ops += [T.ToTensor(), T.Normalize(mean, std)]
    return T.Compose(ops)

def build_train_aug_transform(img, mean, std):
    """Stochastic augmentation used ONLY for TRAIN-time feature extraction (USE_TRAIN_FEATURE_AUG).
    Trash-photo-appropriate transforms that mimic the train->test camera/condition shift: scale+position
    (RandomResizedCrop), lighting/white-balance (ColorJitter), small in-plane rotation, and hflip. NO
    vertical flip (would invert gravity/orientation cues). RandomResizedCrop already emits a full-frame
    square, so there is no SquarePad and no padded region -> features are pooled as a plain mean over all
    patch tokens (which equals masked_mean when nothing is padded), keeping them compatible with the
    clean masked_mean features they're concatenated with for head training."""
    return T.Compose([
        T.RandomResizedCrop(img, scale=(0.65, 1.0), ratio=(0.8, 1.25), interpolation=BICUBIC),
        T.RandomHorizontalFlip(0.5),
        T.RandomApply([T.RandomRotation(12, interpolation=BICUBIC)], p=0.5),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        T.ToTensor(), T.Normalize(mean, std),
    ])

def compute_patch_validity_mask(w, h, img_size, mode="pad", patch=14):
    """Fix B: SquarePad (edge-replicate) fabricates background outside the original image content.
    Mean-pooling over ALL patch tokens then dilutes the signal with replicated-edge tokens, worse
    the more extreme the aspect ratio. This returns a flattened bool grid (True = patch overlaps the
    REAL image content, not the padded/replicated margin) so pooling can down-weight/exclude padding.
    For 'squash'/'crop' modes there is no fabricated content, so every patch is valid."""
    grid = img_size // patch
    if mode != "pad":
        return np.ones(grid * grid, dtype=bool)
    s = max(w, h)
    l = (s - w) // 2; t = (s - h) // 2
    scale = img_size / s
    x0, x1 = l * scale, (l + w) * scale
    y0, y1 = t * scale, (t + h) * scale
    cell = img_size / grid
    gy = np.arange(grid); gx = np.arange(grid)
    cy0, cy1 = gy * cell, (gy + 1) * cell
    cx0, cx1 = gx * cell, (gx + 1) * cell
    valid_y = ~((cy1 <= y0) | (cy0 >= y1))
    valid_x = ~((cx1 <= x0) | (cx0 >= x1))
    mask = valid_y[:, None] & valid_x[None, :]
    return mask.flatten()

class ImgOnlyDS(Dataset):
    def __init__(self, d, tf, lab=True, provide_mask=False, mask_mode=PREPROCESS, mask_img=IMG):
        self.p = d["path"].values; self.y = d["label"].values if lab else None
        self.tf = tf; self.lab = lab
        self.provide_mask = provide_mask; self.mask_mode = mask_mode; self.mask_img = mask_img
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        try:
            img = Image.open(self.p[i]).convert("RGB")
        except Exception:
            if self.lab:
                raise   # train: corrupt files already dropped upstream -> never silently blank a labeled sample
            img = Image.new("RGB", (IMG, IMG), (0, 0, 0))  # test: keep template row order, degrade gracefully
        w, h = img.size
        x = self.tf(img)
        if self.provide_mask:
            mask = torch.from_numpy(compute_patch_validity_mask(w, h, self.mask_img, self.mask_mode))
            if self.lab: return x, int(self.y[i]), mask
            return x, self.p[i], mask
        return (x, int(self.y[i])) if self.lab else (x, self.p[i])

class AttentionPool(nn.Module):
    """Fix B (attn variant): 1 learnable query cross-attends over patch tokens, softmax-masked so
    padded/replicated-edge tokens get exactly zero weight. Cheaper than hand-tuned masking heuristics
    and lets the model learn which patches carry signal. Trained jointly with the unfrozen backbone
    blocks in run_backbone_finetune (fix F) -> avoids ever caching raw per-token features for the
    whole dataset, which would blow up memory (N * num_patches * dim)."""
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, dim) * (dim ** -0.5))
        self.scale = dim ** -0.5
    def forward(self, tokens, mask=None):
        # tokens: [B, N, D], mask: [B, N] bool (True=valid) or None
        q = self.query.expand(tokens.shape[0], -1, -1)                   # [B,1,D]
        attn = (q @ tokens.transpose(1, 2)).squeeze(1) * self.scale       # [B,N]
        if mask is not None:
            attn = attn.masked_fill(~mask, float("-inf"))
            all_masked = (~mask).all(dim=1)
            if all_masked.any():                                        # degenerate: no valid patch -> fall back to uniform
                attn = torch.where(all_masked.unsqueeze(1), torch.zeros_like(attn), attn)
        w = attn.softmax(dim=1).unsqueeze(-1)                             # [B,N,1]
        return (w * tokens).sum(1)                                        # [B,D]

class DINOFeat(nn.Module):
    """Wraps a (frozen or partially-unfrozen, see fix F) DINOv2 backbone; returns L2-normalized
    concat[CLS, pooled(patch tokens)]. Pooling mode is POOLING ("mean" | "masked_mean" | "attn"; fix B).
    autocast lives INSIDE forward so it propagates correctly to nn.DataParallel worker threads."""
    def __init__(self, backbone, num_prefix, pooling=POOLING, attn_pool=None):
        super().__init__()
        self.backbone = backbone; self.num_prefix = num_prefix
        self.pooling = pooling; self.attn_pool = attn_pool
    def forward(self, x, mask=None):
        with torch.amp.autocast("cuda", enabled=(x.device.type == "cuda")):
            t = self.backbone.forward_features(x)          # [B, num_prefix + num_patches, C]
            cls = t[:, 0].float()
            patch_tok = t[:, self.num_prefix:].float()      # [B, num_patches, C]
            if self.pooling == "masked_mean" and mask is not None:
                m = mask.to(patch_tok.dtype).unsqueeze(-1)                    # [B,N,1]
                denom = m.sum(1).clamp_min(1.0)
                patch = (patch_tok * m).sum(1) / denom
            elif self.pooling == "attn" and self.attn_pool is not None:
                patch = self.attn_pool(patch_tok, mask if mask is not None else None)
            else:
                patch = patch_tok.mean(1)
        return torch.cat([F.normalize(cls, dim=-1), F.normalize(patch, dim=-1)], dim=-1)

class ConvFeat(nn.Module):
    """Frozen-probe wrapper for CNN backbones (e.g. ConvNeXt) used in the multi-backbone ensemble.
    forward_features on a CNN returns a spatial map [B, C, H, W] (no token sequence / no CLS), so this
    just global-average-pools + L2-normalizes -> a single feature vector, mirroring what DINOFeat does
    for ViT-style backbones. No mask concept needed (no padding-token dilution issue for a GAP'd CNN)."""
    def __init__(self, backbone):
        super().__init__(); self.backbone = backbone
    def forward(self, x):
        with torch.amp.autocast("cuda", enabled=(x.device.type == "cuda")):
            feat = self.backbone.forward_features(x).float()
            if feat.dim() == 4:            # [B,C,H,W] -> GAP
                feat = feat.mean(dim=(2, 3))
            elif feat.dim() == 3:          # some CNN heads still return a token-ish [B,N,C]; mean over N
                feat = feat.mean(dim=1)
        return F.normalize(feat, dim=-1)

@torch.no_grad()
def extract_features(feat_model, tf, paths, batch=BATCH_EXTRACT, flip=False, use_mask=(POOLING != "mean"),
                      mask_mode=PREPROCESS, mask_img=IMG):
    feat_model.eval()
    ds = ImgOnlyDS(pd.DataFrame({"path": list(paths)}), tf, lab=False,
                   provide_mask=use_mask, mask_mode=mask_mode, mask_img=mask_img)
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=WORKERS, pin_memory=True)
    out = []
    for batch_items in dl:
        if use_mask:
            x, _, mask = batch_items
            mask = mask.to(DEVICE, non_blocking=True)
        else:
            x, _ = batch_items
            mask = None
        x = x.to(DEVICE, non_blocking=True)
        if flip: x = torch.flip(x, dims=[3])
        out.append(feat_model(x, mask).float().cpu().numpy() if mask is not None
                   else feat_model(x).float().cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def extract_features_multiscale(feat_model, mean, std, paths, scales=TTA_SCALES, flip_variants=(False, True),
                                 backbone_id=None, pooling=None, split="test", tag="dinov2"):
    """Fix H: multi-scale TTA (+/-10-15% resize of the square canvas before final resize to IMG),
    combined with hflip. vflip is intentionally NOT used: it can flip the gravity/orientation cues
    that matter for distinguishing e.g. upright vs fallen objects. Each (scale, flip) view goes through
    the on-disk feature cache (see cached_extract_features below)."""
    backbone_id = backbone_id or DINOV2_BACKBONE_ID
    pooling = POOLING if pooling is None else pooling
    all_feats = []
    for s in scales:
        img_s = max(14, int(round(IMG * s / 14)) * 14)   # keep divisible by patch size 14
        tf_s = build_infer_transform(img_s, mean, std, PREPROCESS)
        for flip in flip_variants if USE_TTA else (False,):
            feats = cached_extract_features(tag, backbone_id, img_s, PREPROCESS, pooling, split, paths,
                                            feat_model, tf_s, flip=flip, scale=s,
                                            use_mask=(pooling != "mean"), mask_mode=PREPROCESS, mask_img=img_s)
            all_feats.append(feats)
    return all_feats

# --- on-disk feature cache -------------------------------------------------------------------------
# extract_features() is a pure deterministic function of (frozen backbone weights, transform, which
# images, flip) as long as the backbone isn't fine-tuned mid-run. The key below hashes every knob that
# can change that output, INCLUDING the exact path list -> if dedup/corrupt-drop/noise-label-drop
# changes which rows are in `df`, or any config above changes, the key changes and the cache misses
# automatically (no manual cache-clearing step needed).
def _feat_cache_key(backbone_id, img_size, preprocess, pooling, flip, scale, paths):
    h = hashlib.md5()
    h.update(f"{backbone_id}|{img_size}|{preprocess}|{pooling}|{flip}|{scale}|{len(paths)}".encode())
    h.update("\n".join(paths).encode("utf-8", errors="ignore"))
    return h.hexdigest()[:16]

def cached_extract_features(tag, backbone_id, img_size, preprocess, pooling, split, paths,
                             feat_model, tf, flip=False, scale=1.0, **extract_kwargs):
    if not USE_FEATURE_CACHE:
        return extract_features(feat_model, tf, paths, flip=flip, **extract_kwargs)
    key = _feat_cache_key(backbone_id, img_size, preprocess, pooling, flip, scale, paths)
    fp = os.path.join(FEAT_CACHE_DIR, f"{tag}_{split}_{key}.npy")
    if os.path.exists(fp):
        print(f"[cache] hit -> loading {fp}")
        return np.load(fp)
    print(f"[cache] miss -> extracting fresh (tag={tag} split={split} backbone={backbone_id} img={img_size} "
          f"pooling={pooling} flip={flip} scale={scale} n={len(paths)})")
    feats = extract_features(feat_model, tf, paths, flip=flip, **extract_kwargs)
    np.save(fp, feats)
    return feats

def extract_train_aug_views(feat_model, mean, std, name, backbone_id, img, kind, batch=BATCH_EXTRACT,
                             n_views=TRAIN_AUG_VIEWS):
    """Extract TRAIN_FEATURE_AUG views: for each of n_views, run the stochastic train-aug transform over
    ALL train images and cache the resulting features (frozen into feat_cache on first extraction; a
    cold cache just re-samples equally-valid augmentations). Full-frame aug -> use_mask=False -> plain
    mean pooling for token models (matches the clean masked_mean features when nothing is padded);
    GAP for conv models. Returned as a list of [N, feat_dim] arrays aligned to df row order."""
    views = []
    for v in range(n_views):
        seed_all(SEED + 100 + v)                       # per-view seed -> reproducible & distinct views
        tf_aug = build_train_aug_transform(img, mean, std)
        feats = cached_extract_features(f"{name}_aug", backbone_id, img, PREPROCESS,
                                        ("gap" if kind == "conv" else "mean"), "train",
                                        df["path"].values, feat_model, tf_aug,
                                        flip=False, scale=f"aug{v}", batch=batch, use_mask=False)
        views.append(feats)
    lap(f"[{name}] {n_views} train-aug view(s) extracted")
    return views

class DualHead(nn.Module):
    """LayerNorm -> (linear probe logits + optional MLP logits). Logit-level ensemble of a robust
    linear probe and a small non-linear head."""
    def __init__(self, in_dim, hidden, n_cls, p=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.linear = nn.Linear(in_dim, n_cls)
        self.mlp = (nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(p), nn.Linear(hidden, n_cls))
                    if hidden and hidden > 0 else None)
    def forward(self, x):
        x = self.norm(x); out = self.linear(x)
        if self.mlp is not None: out = out + self.mlp(x)
        return out

def train_head(Xtr, ytr, Xva, yva=None, fold_seed=SEED, lr=DINOV2_HEAD_LR, hidden=DINOV2_HEAD_HIDDEN,
                dropout=HEAD_DROPOUT, wd=HEAD_WD, epochs=DINOV2_HEAD_EPOCHS, use_mixup=USE_FEATURE_MIXUP):
    """Train a DualHead on cached features; returns (best_state, best_val_probs, best_f1).
    fold_seed: fix I — previously always g.manual_seed(SEED) regardless of fold, inconsistent with
    seed_all(SEED+fold) in run_cv_heads. Now each fold's head-training RNG is genuinely per-fold.
    use_mixup: fix A — feature-space mixup (linear interpolation of two cached embeddings + soft-label
    CE) is nearly free since the backbone stays frozen and features are already cached; cheapest
    high-payoff regularizer against the head overfitting to per-sample embedding noise over 40 epochs."""
    in_dim = Xtr.shape[1]
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=DEVICE)
    ytr_t = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=DEVICE)
    head = DualHead(in_dim, hidden, NUM_CLASSES, p=dropout).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    g = torch.Generator(); g.manual_seed(fold_seed)
    samp_w = sampler(ytr, g=g).weights
    loss_fn = cb_focal if USE_CB_FOCAL else smooth_ce
    best_f1, best_probs, best_state = -1, None, None
    for ep in range(epochs):
        head.train()
        order = list(WeightedRandomSampler(samp_w, len(ytr), True, generator=g))
        for i in range(0, len(order), BATCH_HEAD):
            bidx = order[i:i + BATCH_HEAD]
            xb, yb = Xtr_t[bidx], ytr_t[bidx]
            if use_mixup and torch.rand(1, generator=g).item() < FEAT_MIXUP_PROB and len(bidx) > 1:
                lam = float(np.random.default_rng(int(torch.randint(0, 2**31 - 1, (1,), generator=g))).beta(
                    FEAT_MIXUP_ALPHA, FEAT_MIXUP_ALPHA))
                perm = torch.randperm(len(bidx), generator=g, device="cpu").to(DEVICE)
                xb_mix = lam * xb + (1 - lam) * xb[perm]
                out = head(xb_mix)
                y_onehot = F.one_hot(yb, NUM_CLASSES).float()
                y_soft = lam * y_onehot + (1 - lam) * y_onehot[perm]
                y_soft = y_soft * (1 - LS) + LS / NUM_CLASSES   # keep same label smoothing as smooth_ce
                loss = soft_ce(out, y_soft)
            else:
                out = head(xb); loss = loss_fn(out, yb)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        sched.step()
        head.eval()
        with torch.no_grad():
            probs = head(Xva_t).softmax(1).cpu().numpy()
        f1 = macro_f1(yva, probs.argmax(1)) if yva is not None else ep  # if no val labels, keep last
        if f1 > best_f1:
            best_f1, best_probs = f1, probs
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    return best_state, best_probs, best_f1

def head_hpo(feats_all, labels_all, trials=HEAD_HPO_TRIALS, grid=HEAD_HPO_GRID):
    """Fix E: cheap random-search sweep over head LR/hidden/dropout/wd on cached features (seconds/trial).
    CHANGED: instead of sweeping on a hardcoded CV fold (fold 0 was degenerate last run — a single image —
    so every trial scored f1=1.0000 and the sweep picked essentially at random), carve an INDEPENDENT,
    well-sized, group-aware holdout (~1/6 of the data via StratifiedGroupKFold on dup_group, so no near-dup
    leaks across the HPO split either). The winning cfg is then reused for all folds' full training."""
    groups = df["dup_group"].values if "dup_group" in df.columns else np.arange(len(df))
    splitter = StratifiedGroupKFold(6, shuffle=True, random_state=SEED)
    tr_idx, va_idx = next(splitter.split(df, df.label, groups=groups))
    rng = np.random.default_rng(SEED)
    combos = [(lr, h, dp, wd) for lr in grid["lr"] for h in grid["hidden"]
              for dp in grid["dropout"] for wd in grid["wd"]]
    rng.shuffle(combos)
    combos = combos[:trials]
    best_cfg, best_f1 = None, -1
    for (lr, h, dp, wd) in combos:
        _, _, f1 = train_head(feats_all[tr_idx], labels_all[tr_idx], feats_all[va_idx], labels_all[va_idx],
                              fold_seed=SEED, lr=lr, hidden=h, dropout=dp, wd=wd,
                              epochs=max(10, DINOV2_HEAD_EPOCHS // 2))  # shorter budget per trial, cheap sweep
        if f1 > best_f1:
            best_f1, best_cfg = f1, {"lr": lr, "hidden": h, "dropout": dp, "wd": wd}
    print(f"[HPO] {len(combos)} trials on a {len(va_idx)}-img group-aware holdout -> best f1={best_f1:.4f} cfg={best_cfg}")
    return best_cfg

def run_cv_heads(feats_all, labels_all, tag="", head_cfg=None, aug_views=None):
    """aug_views (USE_TRAIN_FEATURE_AUG): optional list of augmented-feature arrays aligned to df. For
    each fold, the TRAINING set becomes clean[train] + each aug_view[train] (labels replicated); the
    VALIDATION set stays CLEAN only -> OOF is never computed on an augmented view, so no leakage and OOF
    stays an honest estimate of clean-image performance."""
    head_cfg = head_cfg or {"lr": DINOV2_HEAD_LR, "hidden": DINOV2_HEAD_HIDDEN, "dropout": HEAD_DROPOUT, "wd": HEAD_WD}
    oof = np.zeros((len(df), NUM_CLASSES), np.float32); filled = np.zeros(len(df), bool); scores = {}
    backbone_label = tag.lstrip("_") if tag else "dinov2"   # bug fix: was hardcoded "dinov2" for every backbone
    for fold in FOLDS_TO_RUN_DINOV2:
        seed_all(SEED + fold)
        tr_idx = df.index[df.fold != fold].values
        va_idx = df.index[df.fold == fold].values
        Xtr, ytr = feats_all[tr_idx], labels_all[tr_idx]
        if aug_views:
            Xtr = np.concatenate([Xtr] + [av[tr_idx] for av in aug_views], axis=0)
            ytr = np.concatenate([ytr] + [labels_all[tr_idx] for _ in aug_views], axis=0)
        state, probs, f1 = train_head(Xtr, ytr,
                                      feats_all[va_idx], labels_all[va_idx], fold_seed=SEED + fold, **head_cfg)
        oof[va_idx] = probs; filled[va_idx] = True; scores[fold] = f1
        torch.save({"state_dict": state, "in_dim": feats_all.shape[1], "hidden": head_cfg["hidden"]},
                   f"{WORK}/head_f{fold}{tag}.pt")
        print(f"  [{backbone_label}] fold{fold} best val_macroF1={f1:.4f}"
              f"{f' (train+{len(aug_views)} aug views)' if aug_views else ''}", flush=True)
    return oof, filled, scores

# --- temperature scaling calibration (fix G) ----------------------------------------------------
def fit_temperature(logits, y, iters=200, lr=0.01):
    """Fits a single scalar T minimizing NLL of softmax(logits/T) on OOF data. Frozen-feature linear
    probes over high-dim L2-normalized inputs tend to be over/under-confident; CLEAN_CONF_TH=0.90 was
    being compared against raw uncalibrated softmax confidence, so the mislabel-flag threshold's real
    meaning was unknown. logits here are recovered as log(probs) (probs already OOF-softmax outputs)."""
    logits_t = torch.tensor(logits, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=lr, max_iter=iters)
    def closure():
        opt.zero_grad()
        T = logT.exp()
        loss = F.cross_entropy(logits_t / T, y_t)
        loss.backward()
        return loss
    opt.step(closure)
    return float(logT.exp().item())

def apply_temperature(probs, T, eps=1e-8):
    logits = np.log(np.clip(probs, eps, 1.0))
    scaled = logits / T
    scaled = scaled - scaled.max(axis=1, keepdims=True)
    e = np.exp(scaled)
    return e / e.sum(axis=1, keepdims=True)

def run_backbone_finetune(backbone, attn_pool, num_prefix, tf, epochs=BACKBONE_FT_EPOCHS,
                          lr_backbone=BACKBONE_LR, batch=BACKBONE_FT_BATCH, fold=0):
    """Fix F (biggest ceiling lever): DINOv2 is pretrained on LVD-142M web images, not close-up trash
    photos (odd backgrounds/lighting/small objects) — a pure frozen linear probe caps out well below
    what a lightly-adapted backbone can reach. Unfreezes only the last UNFREEZE_LAST_N_BLOCKS transformer
    blocks (+final norm) with a discriminative LR far below the head LR, and trains a few epochs with a
    throwaway linear classifier for supervision. Also jointly trains the AttentionPool query (fix B)
    when POOLING=='attn', since it needs gradient signal and this is the only place we do multiple
    forward/backward passes over the backbone anyway — avoids ever caching raw per-token features.
    Discards the throwaway classifier afterward; only the updated backbone/attn_pool weights are kept
    for the single-pass feature caching that follows."""
    if UNFREEZE_LAST_N_BLOCKS <= 0:
        return
    lap(f"partial unfreeze fine-tune: last {UNFREEZE_LAST_N_BLOCKS} blocks, {epochs} epochs, lr={lr_backbone}")
    blocks = backbone.blocks if hasattr(backbone, "blocks") else None
    if blocks is None:
        print("[finetune][WARN] backbone has no .blocks attribute, skipping partial unfreeze"); return
    ft_params = []
    for blk in blocks[-UNFREEZE_LAST_N_BLOCKS:]:
        for p in blk.parameters(): p.requires_grad_(True); ft_params.append(p)
    if hasattr(backbone, "norm"):
        for p in backbone.norm.parameters(): p.requires_grad_(True); ft_params.append(p)

    tr_df = df[df.fold != fold].reset_index(drop=True)
    va_df = df[df.fold == fold].reset_index(drop=True)
    trl = DataLoader(ImgOnlyDS(tr_df, tf, lab=True, provide_mask=(POOLING != "mean")),
                     batch_size=batch, shuffle=True, num_workers=WORKERS, pin_memory=True, drop_last=True)
    val = DataLoader(ImgOnlyDS(va_df, tf, lab=True, provide_mask=(POOLING != "mean")),
                     batch_size=batch * 2, shuffle=False, num_workers=WORKERS, pin_memory=True)

    feat_dim = 2 * backbone.num_features
    clf = nn.Linear(feat_dim, NUM_CLASSES).to(DEVICE)
    param_groups = [{"params": ft_params, "lr": lr_backbone}, {"params": clf.parameters(), "lr": lr_backbone * 100}]
    if attn_pool is not None:
        param_groups.append({"params": attn_pool.parameters(), "lr": lr_backbone * 100})
    opt = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    fm = DINOFeat(backbone, num_prefix, pooling=POOLING, attn_pool=attn_pool).to(DEVICE)
    for ep in range(epochs):
        fm.train(); backbone.train()
        for batch_items in trl:
            if POOLING != "mean":
                x, y, mask = batch_items; mask = mask.to(DEVICE, non_blocking=True)
            else:
                x, y = batch_items; mask = None
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            feat = fm(x, mask)
            out = clf(feat); loss = smooth_ce(out, y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        fm.eval(); backbone.eval()
        ys, ps = [], []
        with torch.no_grad():
            for batch_items in val:
                if POOLING != "mean":
                    x, y, mask = batch_items; mask = mask.to(DEVICE, non_blocking=True)
                else:
                    x, y = batch_items; mask = None
                x = x.to(DEVICE, non_blocking=True)
                feat = fm(x, mask)
                ps.append(clf(feat).softmax(1).cpu().numpy()); ys.append(y.numpy())
        ys, ps = np.concatenate(ys), np.concatenate(ps)
        print(f"  [finetune] ep{ep+1}/{epochs} val_macroF1={macro_f1(ys, ps.argmax(1)):.4f}", flush=True)
    for p in backbone.parameters(): p.requires_grad_(False)   # freeze again for the caching pass
    if attn_pool is not None:
        for p in attn_pool.parameters(): p.requires_grad_(False)
    del clf, opt; gc.collect(); torch.cuda.empty_cache()
    lap("partial unfreeze fine-tune done")

def run_mode_dinov2():
    lap(f"building DINOv2 backbone: {DINOV2_BACKBONE} @ {IMG}px")
    backbone = timm.create_model(DINOV2_BACKBONE, pretrained=True, num_classes=0,
                                 dynamic_img_size=True).to(DEVICE)
    for p in backbone.parameters(): p.requires_grad_(False)
    num_prefix = getattr(backbone, "num_prefix_tokens", 1)
    dc = resolve_data_config({}, model=backbone)
    mean, std = dc["mean"], dc["std"]
    print(f"[dinov2] embed_dim={backbone.num_features} feat_dim={2*backbone.num_features} "
          f"num_prefix_tokens={num_prefix} mean={np.round(mean,3).tolist()} pooling={POOLING}")
    tf = build_infer_transform(IMG, mean, std, PREPROCESS)

    effective_pooling = POOLING
    if POOLING == "attn" and UNFREEZE_LAST_N_BLOCKS <= 0:
        print("[WARN] POOLING='attn' needs run_backbone_finetune() to train its query vector, but "
              "UNFREEZE_LAST_N_BLOCKS<=0 means that pass won't run -> the query would stay randomly "
              "initialized and silently hurt feature quality. Falling back to POOLING='masked_mean' "
              "for this run. Set UNFREEZE_LAST_N_BLOCKS>0 to actually use learned attention pooling.")
        effective_pooling = "masked_mean"

    attn_pool = AttentionPool(backbone.num_features).to(DEVICE) if effective_pooling == "attn" else None
    if UNFREEZE_LAST_N_BLOCKS > 0:
        run_backbone_finetune(backbone, attn_pool, num_prefix, tf)

    feat_model = DINOFeat(backbone, num_prefix, pooling=effective_pooling, attn_pool=attn_pool).to(DEVICE).eval()
    if DEVICE == "cuda" and N_GPU > 1:
        feat_model = nn.DataParallel(feat_model)   # shard extraction across both T4s
        print(f"[dinov2] using DataParallel across {N_GPU} GPUs for extraction")

    lap(f"extracting TRAIN features (single pass, pooling={effective_pooling}; "
        f"backbone may be lightly fine-tuned per fix F)")
    feats_all = cached_extract_features("dinov2", DINOV2_BACKBONE_ID, IMG, PREPROCESS, effective_pooling, "train",
                                        df["path"].values, feat_model, tf, use_mask=(effective_pooling != "mean"))
    labels_all = df["label"].values
    np.save(f"{WORK}/dinov2_train_feats.npy", feats_all)
    lap(f"train features extracted: {feats_all.shape}")

    aug_views = None
    if USE_TRAIN_FEATURE_AUG and "dinov2" in TRAIN_AUG_BACKBONES and TRAIN_AUG_VIEWS > 0:
        lap(f"extracting {TRAIN_AUG_VIEWS} TRAIN-aug view(s) for dinov2 (val->test gap lever)")
        aug_views = extract_train_aug_views(feat_model, mean, std, "dinov2", DINOV2_BACKBONE_ID, IMG, "token")

    head_cfg = None
    if RUN_HEAD_HPO:
        lap("head hyperparameter sweep (fix E)")
        head_cfg = head_hpo(feats_all, labels_all)

    oof, filled, scores = run_cv_heads(feats_all, labels_all, head_cfg=head_cfg, aug_views=aug_views)
    lap("CV done")
    return feat_model, tf, feats_all, labels_all, oof, filled, scores, mean, std, effective_pooling

def run_frozen_backbone(cfg):
    """Generic version of run_mode_dinov2() for the multi-backbone ensemble (user request: try
    convnext_small alongside DINOv2, then ensemble). Same cheap recipe: frozen extract
    (single pass) -> optional head HPO -> DualHead per fold -> OOF. Never fine-tunes the backbone,
    so wall-clock stays small even with 2-3 extra backbones (tiny/small models are cheap to begin
    with). Kept as a separate function from run_mode_dinov2 (some duplication) rather than merging,
    so the already-working DINOv2 path/fine-tune/attn-pool coupling logic stays untouched."""
    name, kind, timm_id, img = cfg["name"], cfg["kind"], cfg["timm_id"], cfg["img"]
    pooling = cfg.get("pooling", "mean") if kind == "token" else "mean"
    b_batch = cfg.get("batch", BATCH_EXTRACT)   # per-backbone extract batch (big batch cuts DP overhead)
    lap(f"[{name}] building backbone: {timm_id} @ {img}px (kind={kind}, batch={b_batch})")
    # dynamic_img_size is a ViT-only kwarg (lets the position embedding interpolate to non-native
    # input sizes); CNN backbones like ConvNeXt don't take it and would raise TypeError if passed.
    if kind == "token":
        backbone = timm.create_model(timm_id, pretrained=True, num_classes=0, dynamic_img_size=True).to(DEVICE)
    else:
        backbone = timm.create_model(timm_id, pretrained=True, num_classes=0).to(DEVICE)
    for p in backbone.parameters(): p.requires_grad_(False)
    dc = resolve_data_config({}, model=backbone)
    mean, std = dc["mean"], dc["std"]
    tf = build_infer_transform(img, mean, std, PREPROCESS)

    if kind == "conv":
        feat_model = ConvFeat(backbone).to(DEVICE).eval()
        use_mask = False
    else:
        num_prefix = getattr(backbone, "num_prefix_tokens", 1)
        feat_model = DINOFeat(backbone, num_prefix, pooling=pooling, attn_pool=None).to(DEVICE).eval()
        use_mask = (pooling != "mean")
    print(f"[{name}] feat_dim={2*backbone.num_features if kind=='token' else backbone.num_features} "
          f"pooling={pooling if kind=='token' else 'GAP'} mean={np.round(mean,3).tolist()}")
    if DEVICE == "cuda" and N_GPU > 1:
        feat_model = nn.DataParallel(feat_model)
        print(f"[{name}] using DataParallel across {N_GPU} GPUs for extraction")

    lap(f"[{name}] extracting TRAIN features (single pass)")
    cache_pooling = pooling if kind == "token" else "gap"
    feats_all = cached_extract_features(name, timm_id, img, PREPROCESS, cache_pooling, "train",
                                        df["path"].values, feat_model, tf, batch=b_batch, use_mask=use_mask,
                                        mask_mode=PREPROCESS, mask_img=img)
    labels_all = df["label"].values
    lap(f"[{name}] train features extracted: {feats_all.shape}")

    aug_views = None
    if USE_TRAIN_FEATURE_AUG and name in TRAIN_AUG_BACKBONES and TRAIN_AUG_VIEWS > 0:
        lap(f"[{name}] extracting {TRAIN_AUG_VIEWS} TRAIN-aug view(s)")
        aug_views = extract_train_aug_views(feat_model, mean, std, name, timm_id, img, kind, batch=b_batch)

    head_cfg = None
    if RUN_HEAD_HPO:
        lap(f"[{name}] head hyperparameter sweep")
        head_cfg = head_hpo(feats_all, labels_all)

    oof, filled, scores = run_cv_heads(feats_all, labels_all, tag=f"_{name}", head_cfg=head_cfg, aug_views=aug_views)
    print(f"  [{name}] fold scores: {{{', '.join(f'{k}: {v:.4f}' for k, v in scores.items())}}}")
    lap(f"[{name}] CV done")
    return {"name": name, "cfg": cfg, "feat_model": feat_model, "tf": tf, "mean": mean, "std": std,
            "use_mask": use_mask, "oof": oof, "filled": filled, "scores": scores}


In [ ]:
# ============================================================
# 5B. PATH: ConvNeXt fine-tune  (kept UNTOUCHED as an optional fallback; only runs if MODE=="convnext")
# ============================================================
class EMA:
    def __init__(self, m, d=EMA_DECAY):
        import copy; self.ema = copy.deepcopy(m).eval(); self.d = d
        for p in self.ema.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, m):
        for e, s in zip(self.ema.state_dict().values(), m.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(self.d).add_(s.detach(), alpha=1 - self.d)
            else: e.copy_(s)

def get_tf(model):
    dc = resolve_data_config({}, model=model); mean, std = dc["mean"], dc["std"]
    tr = create_transform(input_size=IMG, is_training=True, auto_augment=RAND_AUG,
                          re_prob=RE_PROB, re_mode="pixel", interpolation="bicubic", mean=mean, std=std)
    va = create_transform(input_size=IMG, is_training=False, crop_pct=0.95,
                          interpolation="bicubic", mean=mean, std=std)
    return tr, va

def build_convnext():
    return timm.create_model(CONVNEXT_BACKBONE, pretrained=True, num_classes=NUM_CLASSES,
                             drop_rate=0.1, drop_path_rate=0.1).to(DEVICE)

def run_convnext_fold(fold):
    seed_all(SEED + fold)
    tr, va = df[df.fold != fold].reset_index(drop=True), df[df.fold == fold].reset_index(drop=True)
    model = build_convnext(); ttf, vtf = get_tf(model)
    g = torch.Generator(); g.manual_seed(SEED + fold)
    trl = DataLoader(ImgOnlyDS(tr, ttf), batch_size=BATCH, sampler=sampler(tr.label.values, g=g),
                     num_workers=WORKERS, pin_memory=True, drop_last=True, worker_init_fn=seed_worker, generator=g)
    val = DataLoader(ImgOnlyDS(va, vtf), batch_size=BATCH * 2, shuffle=False, num_workers=WORKERS,
                     pin_memory=True, worker_init_fn=seed_worker)
    head = [p for n, p in model.named_parameters() if any(k in n for k in ("head", "classifier", "fc"))]
    back = [p for n, p in model.named_parameters() if not any(k in n for k in ("head", "classifier", "fc"))]
    opt = torch.optim.AdamW([{"params": back, "lr": LR * BB_LR_MULT}, {"params": head, "lr": LR}], weight_decay=WD)
    steps = len(trl); total = EPOCHS * steps; warm = int(WARMUP * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(1, warm) if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda"); ema = EMA(model)
    mix = Mixup(mixup_alpha=MIXUP_A, cutmix_alpha=CUTMIX_A, prob=1.0, switch_prob=0.5,
                mode="batch", label_smoothing=LS, num_classes=NUM_CLASSES)
    best_f1, best_probs, best_ys, best_state = -1, None, None, None
    for ep in range(EPOCHS):
        mixup = None if ep >= EPOCHS - MIXUP_OFF else mix
        model.train(); t0 = time.time()
        for x, y in trl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            if mixup is not None:
                x, ys_soft = mixup(x, y)
                with torch.amp.autocast("cuda"):
                    out = model(x); loss = soft_ce(out, ys_soft)
            else:
                with torch.amp.autocast("cuda"):
                    out = model(x); loss = cb_focal(out, y) if USE_CB_FOCAL else smooth_ce(out, y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
        ema.ema.eval(); probs, ys = [], []
        with torch.no_grad():
            for x, y in val:
                with torch.amp.autocast("cuda"): p = ema.ema(x.to(DEVICE, non_blocking=True)).softmax(1)
                probs.append(p.float().cpu().numpy()); ys.append(y.numpy())
        probs, ys = np.concatenate(probs), np.concatenate(ys); f1 = macro_f1(ys, probs.argmax(1))
        print(f"  fold{fold} ep{ep+1}/{EPOCHS} val_macroF1={f1:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if f1 > best_f1:
            best_f1, best_probs, best_ys = f1, probs, ys
            best_state = {k: v.detach().cpu().clone() for k, v in ema.ema.state_dict().items()}
    torch.save({"state_dict": best_state}, f"{WORK}/ckpt_f{fold}.pt")
    del model, ema; gc.collect(); torch.cuda.empty_cache()
    return va.index.values, best_ys, best_probs, best_f1

def run_mode_convnext():
    oof = np.zeros((len(df), NUM_CLASSES), np.float32); filled = np.zeros(len(df), bool); scores = {}
    for fold in FOLDS_TO_RUN_CONVNEXT:
        print(f"===== FOLD {fold} (convnext) =====", flush=True)
        _, ys, probs, f1 = run_convnext_fold(fold)
        gi = df.index[df.fold == fold].values
        oof[gi] = probs; filled[gi] = True; scores[fold] = f1
    lap("CV done")
    return oof, filled, scores


## 5. Cross-Validation (execution)

Runs the selected `MODE`: per fold, the frozen features are extracted (cached) and the dual head is trained /
validated, producing out-of-fold (OOF) predictions. Also extracts and OOF-scores the extra frozen backbones
(`convnext_small`, `siglip2_l`) for the ensemble. **Verbatim from the original single cell.**


In [ ]:
# ============================================================
# 6. RUN SELECTED MODE
# ============================================================
feat_model = infer_tf = feats_all = labels_all = feat_mean = feat_std = dinov2_pooling_used = None
if MODE == "dinov2_linear":
    feat_model, infer_tf, feats_all, labels_all, oof, filled, scores, feat_mean, feat_std, dinov2_pooling_used = run_mode_dinov2()
elif MODE == "convnext":
    oof, filled, scores = run_mode_convnext()
else:
    raise ValueError(f"unknown MODE={MODE}")

# fix D: dinov2_linear and convnext were mutually exclusive (MODE switch) even though both are fully
# built out. ENSEMBLE_WITH_CONVNEXT runs the convnext path too (a CNN with different inductive bias =
# free diversity gain) and averages its softmax with the dinov2 head ensemble at inference (section 8).
convnext_oof = convnext_filled = convnext_scores = None
if MODE == "dinov2_linear" and ENSEMBLE_WITH_CONVNEXT:
    lap("running convnext path for logit-level ensemble (fix D)")
    convnext_oof, convnext_filled, convnext_scores = run_mode_convnext()
    print("[ensemble] convnext fold scores:", {k: round(v, 4) for k, v in convnext_scores.items()})

# --- multi-backbone frozen-probe ensemble (user request): convnext_small alongside dinov2,
# each producing its own OOF via the identical cheap recipe, later combined in section 7/9. Distinct
# from ENSEMBLE_WITH_CONVNEXT above, which fine-tunes a convnext end-to-end (expensive, off by default);
# this path never fine-tunes anything, so it stays cheap even with 2-3 extra backbones. ------------------
backbone_results = {}
if MODE == "dinov2_linear":
    backbone_results["dinov2"] = {"name": "dinov2", "feat_model": feat_model, "tf": infer_tf,
                                  "mean": feat_mean, "std": feat_std, "use_mask": (POOLING != "mean"),
                                  "oof": oof, "filled": filled, "scores": scores}
    if RUN_EXTRA_BACKBONES and EXTRA_BACKBONES:
        for cfg in EXTRA_BACKBONES:
            try:
                backbone_results[cfg["name"]] = run_frozen_backbone(cfg)
            except Exception as e:
                # protects the <1h automated-run requirement: an extra backbone failing (e.g. a
                # transient HF download issue) skips just that backbone rather than crashing the run.
                print(f"[{cfg['name']}][WARN] failed to build/extract this backbone, SKIPPING it and "
                      f"continuing with the rest of the ensemble. Error: {e}")
                gc.collect(); torch.cuda.empty_cache()
        lap(f"all backbones done: {list(backbone_results.keys())}")
elif MODE == "convnext":
    # wrap the single convnext result the same way, so section 7/9's ensemble code (built for N
    # backbones) also works unchanged for this legacy single-model path (N=1 degenerates cleanly).
    backbone_results["convnext"] = {"name": "convnext", "oof": oof, "filled": filled, "scores": scores}
BACKBONE_NAMES = list(backbone_results.keys())   # fixed order reused by both OOF-tuning (sec.7) and test-time combine (sec.9)

np.save(f"{WORK}/oof.npy", oof)
df[["path", "label", "fold"]].to_csv(f"{WORK}/train_index.csv", index=False)
print("fold scores:", {k: round(v, 4) for k, v in scores.items()})


## 6. OOF Prediction & Ensemble

OOF label-noise diagnostic (temperature calibration + suspect-label export), pairwise backbone
error-diversity diagnostic, OOF-only ensemble selection (weight-tune vs logistic-meta, chosen by honest
K-fold), per-class bias, and similarity-gated kNN retrieval tuning. **Verbatim from the original single cell.**


In [ ]:
# ============================================================
# 6b. LABEL-NOISE DIAGNOSTIC (OOF disagreement) — surfaces likely mislabeled TRAIN images
# ============================================================
calib_temperature = 1.0
if MODE == "dinov2_linear" and filled.any() and RUN_TEMP_SCALING:
    # fix G: fit a single temperature scalar on OOF probs before trusting CLEAN_CONF_TH against them.
    # Frozen linear probes on high-dim L2-normalized features are commonly over-confident; comparing
    # 0.90 against raw uncalibrated softmax meant we didn't actually know how strict that threshold was.
    calib_temperature = fit_temperature(oof[filled], df.label.values[filled])
    print(f"[CALIBRATION] fitted temperature T={calib_temperature:.3f} on OOF "
          f"({'>1 => was over-confident' if calib_temperature > 1 else '<1 => was under-confident'})")
    oof_calibrated = oof.copy()
    oof_calibrated[filled] = apply_temperature(oof[filled], calib_temperature)
else:
    oof_calibrated = oof

if MODE == "dinov2_linear" and filled.any():
    oof_pred = oof.argmax(1); oof_conf = oof_calibrated.max(1)   # confidence read off the CALIBRATED probs (fix G)
    y_all = df.label.values
    suspect = filled & (oof_pred != y_all) & (oof_conf >= CLEAN_CONF_TH)
    print(f"\n[LABELNOISE] suspected mislabels (OOF wrong & calibrated conf>={CLEAN_CONF_TH}): {int(suspect.sum())} / {int(filled.sum())}")
    if suspect.sum() > 0:
        sus = df.loc[suspect, ["path", "label"]].copy()
        sus["oof_pred"] = oof_pred[suspect]; sus["oof_conf"] = oof_conf[suspect].round(4)
        sus["true_name"] = sus["label"].map(CLASS_NAMES); sus["pred_name"] = sus["oof_pred"].map(CLASS_NAMES)
        sus = sus.sort_values("oof_conf", ascending=False)
        sus.to_csv(f"{WORK}/suspect_labels.csv", index=False)
        print("[LABELNOISE] per-(true->pred) counts:\n",
              sus.groupby(["true_name", "pred_name"]).size().sort_values(ascending=False).head(10))
        print("[LABELNOISE] saved suspect_labels.csv (manual spot-check; do NOT blindly trust).")

    if CLEAN_RETRAIN and suspect.sum() > 0 and feats_all is not None:
        keep = ~suspect
        print(f"[LABELNOISE] CLEAN_RETRAIN=True -> dropping {int(suspect.sum())} rows and retraining heads")
        df_bak = df.copy()
        df = df.loc[keep].reset_index(drop=True)
        feats_all = feats_all[keep]; labels_all = labels_all[keep]
        oof2, filled2, scores2 = run_cv_heads(feats_all, labels_all, tag="")  # overwrites head_f*.pt
        oof, filled, scores = np.zeros((len(df), NUM_CLASSES), np.float32), filled2, scores2
        oof = oof2
        print("[LABELNOISE] retrained fold scores:", {k: round(v, 4) for k, v in scores.items()})
        # keep backbone_results["dinov2"] in sync (section 7 reads from the dict, not these bare vars)
        backbone_results["dinov2"]["oof"] = oof
        backbone_results["dinov2"]["filled"] = filled
        backbone_results["dinov2"]["scores"] = scores
        if len(BACKBONE_NAMES) > 1:
            print("[LABELNOISE][WARN] CLEAN_RETRAIN dropped rows from `df`, but the OTHER backbones' "
                  "cached OOF/features were computed on the ORIGINAL (uncleaned) df -> now misaligned "
                  "in length/order. Don't combine CLEAN_RETRAIN with RUN_EXTRA_BACKBONES in the same run "
                  "without re-running run_frozen_backbone(cfg) for each extra backbone AFTER this point.")


In [ ]:
# ============================================================
# 7. OOF ENSEMBLE FIT (train-only, macro-F1 oriented, no test leakage / no Data Uji involved)
# ============================================================
# m = rows filled in EVERY backbone's OOF (should be all rows since every backbone runs all N_FOLDS,
# but computed as an intersection to be safe rather than assuming per-backbone `filled` line up).
m = np.logical_and.reduce([backbone_results[n]["filled"] for n in BACKBONE_NAMES])
yv = df.label.values[m]
P_list = [backbone_results[n]["oof"][m] for n in BACKBONE_NAMES]
n_b, ncls = len(P_list), NUM_CLASSES

# fix M: pairwise backbone error-diversity diagnostic, BEFORE any ensembling. If two backbones are wrong
# on (near-)identical images, no combination method (weight_tune / logistic_meta / class_bias below) can
# rescue those rows -- they are wrong everywhere. Only real, lower error-overlap means there is genuine
# complementary signal for the combination step to exploit. Q-statistic and error-overlap definitions
# follow Kuncheva, "Combining Pattern Classifiers": N11=both correct, N00=both wrong, N10/N01=split.
if n_b >= 2:
    print("\n[DIVERSITY] pairwise backbone error-overlap diagnostic (OOF, before any ensembling):")
    preds_list = [P.argmax(1) for P in P_list]
    correct_list = [pred == yv for pred in preds_list]
    paths_m = df["path"].values[m]
    for i in range(n_b):
        for j in range(i + 1, n_b):
            name_i, name_j = BACKBONE_NAMES[i], BACKBONE_NAMES[j]
            ci, cj = correct_list[i], correct_list[j]
            pi, pj = preds_list[i], preds_list[j]
            disagreement = float((pi != pj).mean())
            n11 = int((ci & cj).sum())       # both correct
            n00 = int((~ci & ~cj).sum())     # both wrong
            n10 = int((ci & ~cj).sum())      # i correct, j wrong
            n01 = int((~ci & cj).sum())      # i wrong, j correct
            either_wrong = n00 + n10 + n01
            error_overlap_pct = n00 / either_wrong if either_wrong else float("nan")
            denom_q = (n11 * n00 + n10 * n01)
            q_stat = (n11 * n00 - n10 * n01) / denom_q if denom_q else float("nan")
            phi = float(np.corrcoef(ci.astype(int), cj.astype(int))[0, 1])
            print(f"  {name_i} vs {name_j}: disagreement_rate={disagreement:.4f} | "
                  f"of {either_wrong} rows where >=1 wrong, BOTH wrong in {n00} ({error_overlap_pct:.1%}) | "
                  f"Q-statistic={q_stat:.3f} | error-correlation(phi)={phi:.3f}")
            print(f"    contingency: both_correct={n11}  both_wrong={n00}  "
                  f"{name_i}_only_correct={n10}  {name_j}_only_correct={n01}")
            interp = ("VERY HIGH overlap -> ensemble ceiling is capped here; combination-method tuning "
                      "won't move the needle much, the bottleneck is backbone diversity, not the combiner"
                      if error_overlap_pct > 0.90 else
                      "high overlap -> limited headroom for stacking/weighting to exploit"
                      if error_overlap_pct > 0.75 else
                      "moderate/low overlap -> real complementary signal; stacking/weighting can help")
            print(f"    interpretation: {interp}")
            if i == 0 and j == 1:
                dis_mask = (pi != pj)
                dis_df = pd.DataFrame({
                    "path": paths_m[dis_mask],
                    "true_name": [CLASS_NAMES[c] for c in yv[dis_mask]],
                    f"{name_i}_pred": [CLASS_NAMES[c] for c in pi[dis_mask]],
                    f"{name_j}_pred": [CLASS_NAMES[c] for c in pj[dis_mask]],
                    f"{name_i}_correct": ci[dis_mask],
                    f"{name_j}_correct": cj[dis_mask],
                })
                dis_df.to_csv(f"{WORK}/backbone_disagreement_{name_i}_vs_{name_j}.csv", index=False)
                print(f"    saved {len(dis_df)} disagreement rows -> backbone_disagreement_{name_i}_vs_{name_j}.csv (manual review)")
    lap("diversity diagnostic done")

def tune_multi(P_list, y, it=3, grid=np.linspace(0.5, 1.8, 27), shrink=0.3):
    """Generalized fix G weight-tune: per-(backbone,class) coordinate-ascent on a weighted SUM of
    backbones' probability matrices (equivalent to a plain average at W==1 before any tuning). shrink
    pulls weights back toward 1.0, a cheap regularizer against overfitting the search to OOF noise."""
    n_b_ = len(P_list); W = np.ones((n_b_, P_list[0].shape[1]))
    combo = lambda W: sum(P_list[b] * W[b] for b in range(n_b_))
    best = macro_f1(y, combo(W).argmax(1))
    for _ in range(it):
        for b in range(n_b_):
            for c in range(P_list[0].shape[1]):
                bw = W[b, c]
                for gv in grid:
                    Wt = W.copy(); Wt[b, c] = gv
                    s = macro_f1(y, combo(Wt).argmax(1))
                    if s > best: best, bw = s, gv
                W[b, c] = bw
    W = 1.0 + (1.0 - shrink) * (W - 1.0)
    return W, macro_f1(y, combo(W).argmax(1))

def fit_logistic_meta(P_list, y, **kw):
    clf = LogisticRegression(max_iter=2000, C=1.0, **kw)
    clf.fit(np.concatenate(P_list, axis=1), y)
    return clf

def apply_weight_tune(P_list, W):
    return sum(P_list[b] * W[b] for b in range(len(P_list)))

def apply_logistic_meta(P_list, clf):
    return clf.predict_proba(np.concatenate(P_list, axis=1))

# fix K: honest method selection via K-fold rather than a single 50/50 split -- one split's
# report-half score has real sampling noise at n_m//2 rows, comparable in size to the ~0.0005-0.001
# gaps that decide weight_tune vs logistic_meta. Averaging the held-out score over K independent folds
# gives a far lower-variance signal for that decision (still 100% OOF-derived, no Data Uji -> compliant).
n_m = int(m.sum())
f1_base_avg = macro_f1(yv, (sum(P_list) / n_b).argmax(1))

# fix J: probation gate. tune_multi / fit_logistic_meta / apply_* are all defined above this point.
def _kfold_best_report(P_sub, folds=5):
    sk = StratifiedKFold(folds, shuffle=True, random_state=SEED)
    wt_s, mt_s = [], []
    for ti, ri in sk.split(np.zeros(len(yv)), yv):
        Pt = [p[ti] for p in P_sub]; Pr = [p[ri] for p in P_sub]
        Wf, _ = tune_multi(Pt, yv[ti]); wt_s.append(macro_f1(yv[ri], apply_weight_tune(Pr, Wf).argmax(1)))
        cf = fit_logistic_meta(Pt, yv[ti]); mt_s.append(macro_f1(yv[ri], apply_logistic_meta(Pr, cf).argmax(1)))
    return max(float(np.mean(wt_s)), float(np.mean(mt_s)))

on_probation = [name in PROBATION_BACKBONES for name in BACKBONE_NAMES]
if any(on_probation) and not all(on_probation):
    keep_idx = [i for i, p in enumerate(on_probation) if not p]
    dropped_names = [n for n, p in zip(BACKBONE_NAMES, on_probation) if p]
    f1_with = _kfold_best_report(P_list)
    f1_without = _kfold_best_report([P_list[i] for i in keep_idx])
    print(f"[ENSEMBLE] probation check {dropped_names}: with={f1_with:.4f}  without(base)={f1_without:.4f}")
    if f1_without >= f1_with:
        print(f"[ENSEMBLE] {dropped_names} REJECTED (no honest K-fold improvement) -> dropped, effective weight=0")
        BACKBONE_NAMES = [BACKBONE_NAMES[i] for i in keep_idx]
        P_list = [P_list[i] for i in keep_idx]
        n_b = len(P_list)
        f1_base_avg = macro_f1(yv, (sum(P_list) / n_b).argmax(1))
    else:
        print(f"[ENSEMBLE] {dropped_names} ACCEPTED (improves honest K-fold report score)")

ENSEMBLE_CV_FOLDS = 5
skf_ens = StratifiedKFold(ENSEMBLE_CV_FOLDS, shuffle=True, random_state=SEED)
wt_fold_scores, meta_fold_scores = [], []
for tune_idx, report_idx in skf_ens.split(np.zeros(n_m), yv):
    P_tune_f = [p[tune_idx] for p in P_list]; P_report_f = [p[report_idx] for p in P_list]
    y_tune_f, y_report_f = yv[tune_idx], yv[report_idx]
    W_f, _ = tune_multi(P_tune_f, y_tune_f)
    wt_fold_scores.append(macro_f1(y_report_f, apply_weight_tune(P_report_f, W_f).argmax(1)))
    clf_f = fit_logistic_meta(P_tune_f, y_tune_f)
    meta_fold_scores.append(macro_f1(y_report_f, apply_logistic_meta(P_report_f, clf_f).argmax(1)))
f1_wt_report = float(np.mean(wt_fold_scores))
f1_meta_report = float(np.mean(meta_fold_scores))

print(f"\n[ENSEMBLE] backbones={BACKBONE_NAMES} | OOF rows={n_m} | plain-average base macroF1={f1_base_avg:.4f}")
print(f"[ENSEMBLE] weight_tune   : {ENSEMBLE_CV_FOLDS}-fold honest report macroF1={f1_wt_report:.4f} (per-fold {np.round(wt_fold_scores,4).tolist()})")
print(f"[ENSEMBLE] logistic_meta : {ENSEMBLE_CV_FOLDS}-fold honest report macroF1={f1_meta_report:.4f} (per-fold {np.round(meta_fold_scores,4).tolist()})")

if ENSEMBLE_METHOD == "weight_tune":
    chosen_method = "weight_tune"
elif ENSEMBLE_METHOD == "logistic_meta":
    chosen_method = "logistic_meta"
else:  # "auto" -> whichever generalizes better on the K-fold honest report score (still OOF-only, compliant)
    chosen_method = "logistic_meta" if f1_meta_report > f1_wt_report else "weight_tune"
print(f"[ENSEMBLE] method={ENSEMBLE_METHOD} -> chosen='{chosen_method}'")

# refit the chosen method on the FULL OOF for the actual submission-facing model, now that the K-fold
# comparison above already gave an honest read on which method to trust.
if chosen_method == "weight_tune":
    W, _ = tune_multi(P_list, yv)
    ensemble_combine = lambda Ps: apply_weight_tune(Ps, W)
    np.save(f"{WORK}/decision_weights.npy", W)
    print(f"[ENSEMBLE] final refit weights (full-OOF):\n{np.round(W,3)}")
else:
    meta_clf = fit_logistic_meta(P_list, yv)
    ensemble_combine = lambda Ps: apply_logistic_meta(Ps, meta_clf)

P_final = ensemble_combine(P_list)
f1_final_full = macro_f1(yv, P_final.argmax(1))

# fix L: per-class probability bias on top of the chosen combiner, tuned to directly target macro-F1
# (argmax on raw probs implicitly optimizes something closer to accuracy, since the metric weights
# classes equally regardless of support). Honestly evaluated via the SAME K-fold split as the method
# choice above (bias fit on each fold's tune portion, scored on that fold's held-out report portion)
# and only kept if it improves the mean held-out score -- otherwise left at 1.0 (a true no-op).
def tune_class_bias(P, y, grid=np.linspace(0.7, 1.4, 29), it=2):
    b = np.ones(P.shape[1])
    best = macro_f1(y, (P * b).argmax(1))
    for _ in range(it):
        for c in range(P.shape[1]):
            bc = b[c]
            for gv in grid:
                bt = b.copy(); bt[c] = gv
                s = macro_f1(y, (P * bt).argmax(1))
                if s > best: best, bc = s, gv
            b[c] = bc
    return b, best

bias_fold_scores, nobias_fold_scores = [], []
for tune_idx, report_idx in skf_ens.split(np.zeros(n_m), yv):
    combo_tune = ensemble_combine([p[tune_idx] for p in P_list])
    combo_report = ensemble_combine([p[report_idx] for p in P_list])
    y_tune_f, y_report_f = yv[tune_idx], yv[report_idx]
    bias_f, _ = tune_class_bias(combo_tune, y_tune_f)
    bias_fold_scores.append(macro_f1(y_report_f, (combo_report * bias_f).argmax(1)))
    nobias_fold_scores.append(macro_f1(y_report_f, combo_report.argmax(1)))
f1_bias_report = float(np.mean(bias_fold_scores))
f1_nobias_report = float(np.mean(nobias_fold_scores))
print(f"[ENSEMBLE] class_bias    : {ENSEMBLE_CV_FOLDS}-fold honest report macroF1={f1_bias_report:.4f}  (no-bias={f1_nobias_report:.4f})")
USE_CLASS_BIAS = f1_bias_report > f1_nobias_report + 1e-4
if USE_CLASS_BIAS:
    class_bias, _ = tune_class_bias(P_final, yv)
    print(f"[ENSEMBLE] class_bias ACCEPTED (improves honest report score) -> final bias={np.round(class_bias,3)}")
else:
    class_bias = np.ones(NUM_CLASSES)
    print(f"[ENSEMBLE] class_bias REJECTED (no honest improvement) -> left at 1.0 (no-op)")
P_final_biased = P_final * class_bias
f1_final_full_biased = macro_f1(yv, P_final_biased.argmax(1))

print(f"\nOOF macroF1: plain-average={f1_base_avg:.4f} | chosen-method(full-OOF refit)={f1_final_full:.4f} | +class_bias={f1_final_full_biased:.4f}")
print(classification_report(yv, P_final_biased.argmax(1), target_names=list(CLASS_NAMES.values()), digits=4))
cm = confusion_matrix(yv, P_final_biased.argmax(1))
print("confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(cm, index=list(CLASS_NAMES.values()), columns=list(CLASS_NAMES.values())))
# fix N: kNN retrieval blend. Pure-numpy cosine soft-vote in the dinov2 feature space (feats_all/yv),
# similarity-gated so it only acts where retrieval is reliable and is a strict no-op otherwise.
def knn_probs(Q, R, R_labels, k=KNN_K, power=KNN_POWER, n_cls=NUM_CLASSES, chunk=512):
    Qn = Q / np.clip(np.linalg.norm(Q, axis=1, keepdims=True), 1e-8, None)
    Rn = R / np.clip(np.linalg.norm(R, axis=1, keepdims=True), 1e-8, None)
    Rl = np.asarray(R_labels)
    out = np.zeros((len(Q), n_cls), np.float32); top1 = np.zeros(len(Q), np.float32)
    for i in range(0, len(Q), chunk):
        sim = Qn[i:i + chunk] @ Rn.T
        top1[i:i + chunk] = sim.max(1)
        kk = min(k, sim.shape[1])
        idx = np.argpartition(-sim, kk - 1, axis=1)[:, :kk]
        rows = np.arange(sim.shape[0])[:, None]
        w = np.clip(sim[rows, idx], 0, None) ** power
        lab = Rl[idx]
        for cl in range(n_cls):
            out[i:i + chunk, cl] = (w * (lab == cl)).sum(1)
        out[i:i + chunk] /= np.clip(out[i:i + chunk].sum(1, keepdims=True), 1e-8, None)
    return out, top1

def knn_gate(sim, lo=KNN_SIM_LO, hi=KNN_SIM_HI):
    return np.clip((sim - lo) / max(hi - lo, 1e-8), 0.0, 1.0)

def knn_gated_blend(model_p, knn_p, sim, alpha):
    a = (alpha * knn_gate(sim))[:, None]
    return (1 - a) * model_p + a * knn_p

KNN_ALPHA = 0.0   # tuned below on OOF leave-fold-out; stays 0 (exact no-op) if retrieval does not help
if RUN_KNN_RETRIEVAL:
    try:
        folds_m = df["fold"].values[m]
        feats_m = feats_all[m]
        oof_knn = np.zeros((len(yv), NUM_CLASSES), np.float32); oof_sim = np.zeros(len(yv), np.float32)
        for fdv in np.unique(folds_m):
            q = folds_m == fdv
            pk, sk = knn_probs(feats_m[q], feats_m[~q], yv[~q])
            oof_knn[q] = pk; oof_sim[q] = sk
        f1_model = macro_f1(yv, P_final_biased.argmax(1))
        best_a, best_f1 = 0.0, f1_model
        for a in np.linspace(0.0, 0.9, 19):
            f1a = macro_f1(yv, knn_gated_blend(P_final_biased, oof_knn, oof_sim, float(a)).argmax(1))
            if f1a > best_f1 + 1e-6:
                best_f1, best_a = f1a, float(a)
        KNN_ALPHA = best_a
        n_gate = int((oof_sim >= KNN_SIM_HI).sum())
        print(f"[KNN] leave-fold-out OOF: model={f1_model:.4f} | knn-only={macro_f1(yv, oof_knn.argmax(1)):.4f} | "
              f"best gated-blend={best_f1:.4f} @ alpha={KNN_ALPHA:.2f} | {n_gate}/{len(yv)} OOF rows at full gate (sim>={KNN_SIM_HI})")
        if KNN_ALPHA == 0.0:
            print("[KNN] alpha=0 -> retrieval left OFF (no honest OOF improvement); submission unchanged")
    except Exception as e:
        KNN_ALPHA = 0.0
        print(f"[KNN][WARN] OOF tuning failed ({e}) -> KNN_ALPHA=0 (no-op)")

lap("OOF ensemble fit done")


## 7. Advanced Post-processing Analysis (NEW)

**New, read-only** analysis built on the OOF arrays (`oof`, `oof_calibrated`, `filled`, `P_final`,
`P_final_biased`, `yv`, `m`) and cached DINOv2 features (`feats_all`) produced by sections 5–6. Metrics use
the pipeline's **final biased decision** (`P_final_biased`) so they match the model that ships. Nothing here
retrains a head or writes to `submission.csv`.


### 7.1 OOF Prediction Summary

In [ ]:
# 7.1 OOF prediction summary (read-only)
print(f"Backbones built             : {list(backbone_results.keys())}")
print(f"Backbones in final ensemble : {BACKBONE_NAMES}")
print(f"Chosen ensemble method      : {chosen_method}")
print(f"Class-bias applied          : {bool(USE_CLASS_BIAS)}  -> {np.round(class_bias, 3).tolist()}")
print(f"kNN retrieval alpha         : {KNN_ALPHA:.2f}  ({'ON' if KNN_ALPHA > 0 else 'OFF / no-op'})")
print(f"OOF rows scored             : {n_m} / {len(df)}")
print(f"Plain-average macro-F1      : {f1_base_avg:.4f}")
print(f"Chosen-method macro-F1      : {f1_final_full:.4f}  (full-OOF refit)")
print(f"+ class-bias  macro-F1      : {f1_final_full_biased:.4f}")
for _n in backbone_results:
    _sc = {k: round(v, 4) for k, v in backbone_results[_n]["scores"].items()}
    print(f"  fold macro-F1 [{_n:<13}]: {_sc}")


### 7.2 Per-class Metrics

In [ ]:
# 7.2 Per-class metrics on the pipeline's FINAL biased OOF decision (so they match the shipped model)
from sklearn.metrics import precision_recall_fscore_support, balanced_accuracy_score, matthews_corrcoef

y_pred_final = P_final_biased.argmax(1)
prec, rec, f1c, support = precision_recall_fscore_support(yv, y_pred_final, labels=list(range(NUM_CLASSES)))
per_class_metrics = pd.DataFrame({
    "class": [CLASS_NAMES[c] for c in range(NUM_CLASSES)],
    "precision": prec.round(4), "recall": rec.round(4), "f1": f1c.round(4), "support": support,
})
per_class_metrics["balanced_accuracy"] = round(balanced_accuracy_score(yv, y_pred_final), 4)
per_class_metrics["mcc_overall"] = round(matthews_corrcoef(yv, y_pred_final), 4)
display(per_class_metrics)


### 7.3 Confusion Matrix

In [ ]:
# 7.3 Confusion matrix (counts + row-normalized) on the final biased OOF decision
from sklearn.metrics import ConfusionMatrixDisplay

cm_raw = confusion_matrix(yv, y_pred_final)
cm_norm = confusion_matrix(yv, y_pred_final, normalize="true")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(cm_raw, display_labels=list(CLASS_NAMES.values())).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix (counts)")
ConfusionMatrixDisplay(cm_norm, display_labels=list(CLASS_NAMES.values())).plot(ax=axes[1], colorbar=False, values_format=".2f")
axes[1].set_title("Confusion matrix (row-normalized)")
plt.tight_layout(); plt.savefig(f"{WORK}/oof_confusion_matrix.png", dpi=120); plt.show()
per_class_error = 1 - np.diag(cm_norm)
display(pd.DataFrame({"class": list(CLASS_NAMES.values()), "error_rate": per_class_error.round(4)}))


### 7.4 Calibration Analysis

In [ ]:
def calibration_stats(probs, y_true, n_bins=15):
    conf = probs.max(1); pred = probs.argmax(1); correct = (pred == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_acc, bin_conf, bin_count = [], [], []
    ece, mce = 0.0, 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        sel = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if sel.sum() == 0:
            bin_acc.append(np.nan); bin_conf.append((lo+hi)/2); bin_count.append(0); continue
        acc_i, conf_i = correct[sel].mean(), conf[sel].mean()
        bin_acc.append(acc_i); bin_conf.append(conf_i); bin_count.append(int(sel.sum()))
        gap = abs(acc_i - conf_i)
        ece += (sel.sum() / len(conf)) * gap
        mce = max(mce, gap)
    onehot = np.eye(NUM_CLASSES)[y_true]
    brier = float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))
    return {"bin_edges": bins, "bin_acc": np.array(bin_acc), "bin_conf": np.array(bin_conf),
            "bin_count": np.array(bin_count), "ece": ece, "mce": mce, "brier": brier}

cal = calibration_stats(oof_calibrated[filled], df.label.values[filled])
print(f"Expected Calibration Error (ECE): {cal['ece']:.4f}")
print(f"Maximum Calibration Error (MCE) : {cal['mce']:.4f}")
print(f"Brier score                     : {cal['brier']:.4f}")
print(f"(fitted temperature T={calib_temperature:.3f} from section 6)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot([0,1],[0,1],"k--", label="perfect calibration")
axes[0].bar(cal["bin_conf"], cal["bin_acc"], width=1/15, alpha=0.6, edgecolor="k", label="model")
axes[0].set_xlabel("confidence"); axes[0].set_ylabel("accuracy"); axes[0].set_title("Reliability diagram"); axes[0].legend()
axes[1].hist(oof_calibrated[filled].max(1), bins=30, color="#4C72B0")
axes[1].set_xlabel("max softmax confidence"); axes[1].set_title("Confidence histogram")
plt.tight_layout(); plt.savefig(f"{WORK}/oof_calibration.png", dpi=120); plt.show()


### 7.5 Cross-validation Stability (per backbone)

In [ ]:
# 7.5 Cross-validation stability, per backbone (mean/std of per-fold macro-F1)
rows = []
for _n, _res in backbone_results.items():
    fs = pd.Series(_res["scores"], dtype=float).sort_index()
    if len(fs) == 0:
        continue
    mu = fs.mean(); sd = fs.std(ddof=1) if len(fs) > 1 else 0.0
    ci = 1.96 * sd / np.sqrt(len(fs)) if len(fs) > 1 else 0.0
    rows.append({"backbone": _n, "mean_f1": round(mu, 4), "std_f1": round(float(sd), 4),
                 "ci95_lo": round(mu - ci, 4), "ci95_hi": round(mu + ci, 4),
                 "n_folds": len(fs), "in_ensemble": _n in BACKBONE_NAMES})
display(pd.DataFrame(rows))

dino = pd.Series(backbone_results["dinov2"]["scores"], dtype=float).sort_index()
mean_f1, std_f1 = dino.mean(), (dino.std(ddof=1) if len(dino) > 1 else 0.0)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(dino.index.astype(str), dino.values, color="#4C72B0")
ax.axhline(mean_f1, color="red", linestyle="--", label=f"mean={mean_f1:.4f}")
ax.fill_between([-0.5, len(dino) - 0.5], mean_f1 - std_f1, mean_f1 + std_f1, color="red", alpha=0.1, label="±1 std")
ax.set_xlabel("fold"); ax.set_ylabel("macro F1"); ax.set_title("dinov2 head CV stability"); ax.legend()
plt.tight_layout(); plt.savefig(f"{WORK}/cv_stability.png", dpi=120); plt.show()


### 7.6 Error Analysis

In [ ]:
# 7.6 Error analysis on the FINAL biased OOF decision (builds `wrong`, reused by section 10)
idx_full = np.arange(len(df))[m]
err_df = df.iloc[idx_full].copy()
err_df["pred"] = y_pred_final
err_df["conf"] = P_final_biased.max(1)
err_df["correct"] = err_df["label"].values == err_df["pred"].values
err_df["pred_name"] = err_df["pred"].map(CLASS_NAMES)
err_df["true_name"] = err_df["label"].map(CLASS_NAMES)

wrong = err_df[~err_df["correct"]].copy()
median_conf = err_df["conf"].median()
wrong["conf_bucket"] = np.where(wrong["conf"] >= median_conf, "high_confidence_error", "low_confidence_error")
print(f"Total OOF errors: {len(wrong)} / {len(err_df)}  ({len(wrong) / max(len(err_df), 1) * 100:.2f}%)")
if len(wrong):
    display(wrong.groupby(["true_name", "pred_name"]).size().sort_values(ascending=False).rename("count").reset_index())
    display(wrong["conf_bucket"].value_counts())
    hi = wrong.sort_values("conf", ascending=False).head(6)
    show_gallery(list(zip(hi["path"], hi["true_name"] + "->" + hi["pred_name"] + " (" + hi["conf"].round(2).astype(str) + ")")),
                 title="High-confidence OOF errors")
    lo = wrong.sort_values("conf", ascending=True).head(6)
    show_gallery(list(zip(lo["path"], lo["true_name"] + "->" + lo["pred_name"] + " (" + lo["conf"].round(2).astype(str) + ")")),
                 title="Low-confidence OOF errors")


### 7.7 Long-tail / Hidden Subtype Analysis

In [ ]:
from sklearn.cluster import KMeans

if "feats_all" in globals() and feats_all is not None:
    worst_class_name = per_class_metrics.loc[per_class_metrics["f1"].idxmin(), "class"]
    worst_class_id = {v: k for k, v in CLASS_NAMES.items()}[worst_class_name]
    print(f"Focus class (lowest OOF F1): {worst_class_name}")

    focus_idx = df.index[(df.label == worst_class_id)].values
    focus_feats = feats_all[focus_idx]
    focus_df = df.loc[focus_idx].copy()
    focus_df["oof_pred"] = oof.argmax(1)[focus_idx]
    focus_df["oof_conf"] = oof.max(1)[focus_idx]
    focus_df["correct"] = focus_df["oof_pred"] == focus_df["label"]

    k = min(6, max(2, len(focus_idx) // 50))   # simple fixed-k KMeans on the frozen embeddings
    focus_df["cluster"] = KMeans(k, random_state=SEED, n_init=10).fit_predict(focus_feats)

    cluster_report = focus_df.groupby("cluster").agg(
        n_samples=("path", "size"), accuracy=("correct", "mean"),
        mean_conf=("oof_conf", "mean")).reset_index()
    cluster_report["error_rate"] = 1 - cluster_report["accuracy"]
    cluster_report = cluster_report.sort_values("error_rate", ascending=False)
    display(cluster_report)

    # one simple bar chart: error rate per cluster (the whole point of this analysis)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(cluster_report["cluster"].astype(str), cluster_report["error_rate"], color="#C44E52")
    ax.set_xlabel("cluster id"); ax.set_ylabel("OOF error rate")
    ax.set_title(f"{worst_class_name}: error rate by embedding cluster (n={len(focus_idx)})")
    plt.tight_layout(); plt.savefig(f"{WORK}/longtail_cluster_error.png", dpi=120); plt.show()

    worst_cluster = cluster_report.iloc[0]["cluster"]
    examples = focus_df[focus_df.cluster == worst_cluster].sample(min(6, (focus_df.cluster == worst_cluster).sum()), random_state=SEED)
    show_gallery(list(zip(examples["path"], [worst_class_name] * len(examples))),
                 title=f"{worst_class_name}: highest-error cluster ({worst_cluster})")
else:
    print("[placeholder] Needs `feats_all` (cached DINOv2 features from section 5's run_mode_dinov2()) "
          "in memory. Run sections 4-6 first, then re-run this cell.")


### 7.8 Embedding Visualization (PCA)

In [ ]:
# 7.8 DINOv2 embedding 2D projection (PCA) — OOF only (train/test overlay is in section 10.3)
from sklearn.decomposition import PCA

if "feats_all" in globals() and feats_all is not None:
    emb2d = PCA(n_components=2, random_state=SEED).fit_transform(feats_all)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(emb2d[:, 0], emb2d[:, 1], c=df.label.values, cmap="tab10", s=4, alpha=0.6)
    axes[0].set_title("DINOv2 embedding (PCA) colored by class")
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.tab10(i / 10), label=n)
               for i, n in CLASS_NAMES.items()]
    axes[0].legend(handles=handles)
    correct_mask = np.full(len(df), np.nan)
    correct_mask[m] = (P_final_biased.argmax(1) == df.label.values[m]).astype(float)
    axes[1].scatter(emb2d[:, 0], emb2d[:, 1], c=correct_mask, cmap="RdYlGn", s=4, alpha=0.6)
    axes[1].set_title("Same embedding: OOF correct (green) vs wrong (red)")
    plt.tight_layout(); plt.savefig(f"{WORK}/embedding_pca.png", dpi=120); plt.show()
else:
    print("[note] feats_all not in memory — run sections 4-6 first.")


### 7.9 Ensemble Method Comparison (NEW)

In [ ]:
# 7.9 Ensemble strategy comparison (all scores are OOF-derived, computed by the pipeline in section 6)
methods = {
    "plain average": f1_base_avg,
    "weight_tune (kfold)": f1_wt_report,
    "logistic_meta (kfold)": f1_meta_report,
    "chosen, no bias (kfold)": f1_nobias_report,
    "chosen + class_bias (kfold)": f1_bias_report,
    "FINAL (full-OOF + bias)": f1_final_full_biased,
}
display(pd.DataFrame({"method": list(methods.keys()), "macro_f1": [round(v, 4) for v in methods.values()]}))
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(range(len(methods)), list(methods.values()), color="#4C72B0")
bars[-1].set_color("#C44E52")
ax.set_xticks(range(len(methods))); ax.set_xticklabels(list(methods.keys()), rotation=25, ha="right")
ax.set_ylabel("macro F1"); ax.set_title(f"Ensemble strategy comparison (chosen: {chosen_method}, bias={'on' if USE_CLASS_BIAS else 'off'})")
ax.set_ylim(min(methods.values()) - 0.01, max(methods.values()) + 0.005)
for i, v in enumerate(methods.values()):
    ax.text(i, v, f"{v:.4f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig(f"{WORK}/ensemble_method_comparison.png", dpi=120); plt.show()


### 7.10 Backbone Diversity & Agreement (NEW)

In [ ]:
# 7.10 Backbone diversity & agreement on the m-masked OOF (why the ensemble can help)
names = list(backbone_results.keys())
preds_by = {n: backbone_results[n]["oof"][m].argmax(1) for n in names}
corr_by = {n: (preds_by[n] == yv) for n in names}
bb_f1 = {n: macro_f1(yv, preds_by[n]) for n in names}
display(pd.DataFrame({"backbone": names,
                      "oof_macro_f1": [round(bb_f1[n], 4) for n in names],
                      "in_ensemble": [n in BACKBONE_NAMES for n in names]}))

if len(names) >= 2:
    K = len(names)
    agree = np.zeros((K, K)); eover = np.full((K, K), np.nan)
    for a in range(K):
        for b in range(K):
            agree[a, b] = float((preds_by[names[a]] == preds_by[names[b]]).mean())
            if a != b:
                ca, cb = corr_by[names[a]], corr_by[names[b]]
                either = int((~(ca & cb)).sum()); both_wrong = int((~ca & ~cb).sum())
                eover[a, b] = both_wrong / either if either else np.nan
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    im0 = axes[0].imshow(agree, vmin=0.9, vmax=1.0, cmap="viridis")
    axes[0].set_title("Prediction agreement rate")
    im1 = axes[1].imshow(np.nan_to_num(eover), vmin=0, vmax=1, cmap="magma")
    axes[1].set_title("Error overlap (both wrong / ≥1 wrong)")
    for ax, mat, fmt in [(axes[0], agree, "{:.3f}"), (axes[1], eover, "{:.2f}")]:
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        ax.set_xticklabels(names, rotation=30, ha="right"); ax.set_yticklabels(names)
        for a in range(K):
            for b in range(K):
                v = mat[a, b]
                if not np.isnan(v):
                    ax.text(b, a, fmt.format(v), ha="center", va="center", color="w", fontsize=8)
    fig.colorbar(im0, ax=axes[0], fraction=0.046); fig.colorbar(im1, ax=axes[1], fraction=0.046)
    plt.tight_layout(); plt.savefig(f"{WORK}/backbone_diversity.png", dpi=120); plt.show()
    print("Lower error-overlap => more complementary backbones => more headroom for stacking/weighting.")
else:
    print("[note] only one backbone available -> no pairwise diversity to show.")


### 7.11 kNN Retrieval Contribution (NEW)

In [ ]:
# 7.11 kNN retrieval contribution (tuned on OOF in section 6; strict no-op if alpha==0)
print(f"KNN_ALPHA (OOF-tuned)  : {KNN_ALPHA:.3f}  ({'active' if KNN_ALPHA > 0 else 'inactive / exact no-op'})")
print(f"KNN gate window        : sim in [{KNN_SIM_LO}, {KNN_SIM_HI}]  (k={KNN_K}, power={KNN_POWER})")
if RUN_KNN_RETRIEVAL and "oof_sim" in globals():
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(oof_sim, bins=40, color="#55A868")
    ax.axvline(KNN_SIM_LO, color="k", ls="--", label=f"gate lo={KNN_SIM_LO}")
    ax.axvline(KNN_SIM_HI, color="r", ls="--", label=f"gate hi={KNN_SIM_HI}")
    ax.set_xlabel("OOF top-1 neighbour cosine similarity"); ax.set_ylabel("count")
    ax.set_title("kNN retrieval reliability gate on OOF"); ax.legend()
    plt.tight_layout(); plt.savefig(f"{WORK}/knn_gate.png", dpi=120); plt.show()
    print(f"OOF rows at full gate (sim>={KNN_SIM_HI}): {int((oof_sim >= KNN_SIM_HI).sum())} / {len(oof_sim)}")
else:
    print("[note] kNN OOF diagnostics (oof_sim) not in memory or retrieval disabled.")


### 7.12 Saliency / Explainability

In [ ]:
# 7.12 Input-gradient saliency (dependency-free ViT explainability; forward+backward for viz only)
try:
    if "feat_model" in globals() and feat_model is not None and MODE == "dinov2_linear" and os.path.exists(f"{WORK}/head_f0.pt"):
        ckpt = torch.load(f"{WORK}/head_f0.pt", map_location=DEVICE)
        head0 = DualHead(ckpt["in_dim"], ckpt.get("hidden", DINOV2_HEAD_HIDDEN), NUM_CLASSES).to(DEVICE)
        head0.load_state_dict(ckpt["state_dict"]); head0.eval()
        fm = feat_model.module if isinstance(feat_model, nn.DataParallel) else feat_model
        sample_row = df.sample(1, random_state=SEED).iloc[0]
        with _Image.open(sample_row.path) as im:
            x = infer_tf(im.convert("RGB")).unsqueeze(0).to(DEVICE).requires_grad_(True)
        out = head0(fm(x)); pred_cls = int(out.argmax(1).item())
        out[0, pred_cls].backward()
        saliency = x.grad.abs().amax(dim=1)[0].cpu().numpy()
        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        axes[0].imshow(_Image.open(sample_row.path).convert("RGB"))
        axes[0].set_title(f"Input (true={CLASS_NAMES[sample_row.label]}, pred={CLASS_NAMES[pred_cls]})"); axes[0].axis("off")
        axes[1].imshow(saliency, cmap="hot"); axes[1].set_title("Input-gradient saliency"); axes[1].axis("off")
        plt.tight_layout(); plt.savefig(f"{WORK}/saliency_example.png", dpi=120); plt.show()
    else:
        print("[placeholder] needs feat_model/infer_tf from section 5 and a saved fold-0 head.")
except Exception as e:
    print(f"[saliency] skipped (visualization-only, does not affect the pipeline): {e}")


### 7.13 Robustness Testing

In [ ]:
# 7.13 Robustness: fold-0 head confidence under light, inference-only perturbations (viz only)
def perturb(im, kind):
    if kind == "brightness":
        return T.functional.adjust_brightness(im, 1.4)
    if kind == "blur":
        return im.filter(__import__("PIL").ImageFilter.GaussianBlur(2))
    if kind == "rotate":
        return im.rotate(8, resample=_Image.BICUBIC, fillcolor=(128, 128, 128))
    return im

try:
    if "feat_model" in globals() and feat_model is not None and MODE == "dinov2_linear" and os.path.exists(f"{WORK}/head_f0.pt"):
        ckpt = torch.load(f"{WORK}/head_f0.pt", map_location=DEVICE)
        head0 = DualHead(ckpt["in_dim"], ckpt.get("hidden", DINOV2_HEAD_HIDDEN), NUM_CLASSES).to(DEVICE)
        head0.load_state_dict(ckpt["state_dict"]); head0.eval()
        fm = feat_model.module if isinstance(feat_model, nn.DataParallel) else feat_model
        rows = []
        for _, r in df.sample(min(30, len(df)), random_state=SEED).iterrows():
            with _Image.open(r.path) as im0:
                im0 = im0.convert("RGB")
                with torch.no_grad():
                    clean = head0(fm(infer_tf(im0).unsqueeze(0).to(DEVICE))).softmax(1).max().item()
                for kind in ["brightness", "blur", "rotate"]:
                    with torch.no_grad():
                        pc = head0(fm(infer_tf(perturb(im0, kind)).unsqueeze(0).to(DEVICE))).softmax(1).max().item()
                    rows.append({"perturbation": kind, "delta": pc - clean})
        avg = pd.DataFrame(rows).groupby("perturbation")["delta"].mean()
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(avg.index, avg.values, color="#8172B2"); ax.axhline(0, color="k", linewidth=0.8)
        ax.set_ylabel("avg. confidence change (perturbed - clean)")
        ax.set_title("Robustness: confidence drop under light perturbations")
        plt.tight_layout(); plt.savefig(f"{WORK}/robustness_test.png", dpi=120); plt.show()
    else:
        print("[placeholder] needs feat_model/infer_tf from section 5 and a saved fold-0 head.")
except Exception as e:
    print(f"[robustness] skipped (visualization-only, does not affect the pipeline): {e}")


## 8. Prediction (test inference)

Test feature extraction (multi-scale + hflip TTA), fold-ensemble head inference for every backbone, and the
train↔test near-duplicate cosine-similarity leakage check. **Verbatim from the original single cell.**


In [ ]:
# ============================================================
# 8. INFERENCE (fold-ensemble + hflip TTA) — driven by the official template CSV
# ============================================================
sub = pd.read_csv(SUBT); ids = sub["id"].tolist()
print(f"[infer] template rows: {len(ids)}")

def resolve_test_path(i):
    # robust to extension: try common ones, else glob {i}.*
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".webp", ".bmp"):
        p = os.path.join(TEST, f"{i}{ext}")
        if os.path.exists(p): return p
    hits = glob.glob(os.path.join(TEST, f"{i}.*"))
    return hits[0] if hits else os.path.join(TEST, f"{i}.jpg")

tpaths = [resolve_test_path(i) for i in ids]
missing = [p for p in tpaths if not os.path.exists(p)]
if missing: print(f"[infer][WARN] {len(missing)} test paths not found, e.g. {missing[:3]}")

if MODE == "dinov2_linear":
    lap("extracting TEST features (multi-scale + hflip TTA, fix H)")
    test_feats = cached_extract_features("dinov2", DINOV2_BACKBONE_ID, IMG, PREPROCESS, dinov2_pooling_used, "test",
                                         tpaths, feat_model, infer_tf,
                                         use_mask=(dinov2_pooling_used != "mean"))  # base scale, no flip — kept for the train/test dedup check below
    views = extract_features_multiscale(feat_model, feat_mean, feat_std, tpaths,
                                        backbone_id=DINOV2_BACKBONE_ID, pooling=dinov2_pooling_used,
                                        split="test", tag="dinov2")
    np.save(f"{WORK}/dinov2_test_feats.npy", test_feats)

    tprobs = np.zeros((len(ids), NUM_CLASSES), np.float32); nu = 0
    head_ckpts = [(fold, f"{WORK}/head_f{fold}.pt") for fold in FOLDS_TO_RUN_DINOV2]
    for fold, cp in head_ckpts:
        if not os.path.exists(cp): continue
        ckpt = torch.load(cp, map_location=DEVICE)
        head = DualHead(ckpt["in_dim"], ckpt.get("hidden", DINOV2_HEAD_HIDDEN), NUM_CLASSES).to(DEVICE)
        head.load_state_dict(ckpt["state_dict"]); head.eval()
        with torch.no_grad():
            for v in views:
                vt = torch.tensor(v, dtype=torch.float32, device=DEVICE)
                tprobs += head(vt).softmax(1).cpu().numpy()
                nu += 1
    tprobs /= max(nu, 1)

    # fix C: near-dup / leakage check between TRAIN and TEST via cosine similarity on cached DINOv2
    # features — nearly free since both are already extracted. Waste-image competition datasets are
    # often derived from the same public corpora (e.g. TrashNet-style), so a real test image can be a
    # near-duplicate of a train image. If so, override the model's prediction with the train label.
    if RUN_TRAIN_TEST_DEDUP_CHECK:
        lap("train<->test near-dup cosine-similarity check (fix C)")
        train_feat_n = feats_all / np.clip(np.linalg.norm(feats_all, axis=1, keepdims=True), 1e-8, None)
        test_feat_n = test_feats / np.clip(np.linalg.norm(test_feats, axis=1, keepdims=True), 1e-8, None)
        train_labels_arr = labels_all
        n_overridden = 0
        CHUNK = 512
        override_label = np.full(len(test_feat_n), -1, dtype=int)
        best_sim = np.zeros(len(test_feat_n), dtype=np.float32)
        for i in range(0, len(test_feat_n), CHUNK):
            sim = test_feat_n[i:i + CHUNK] @ train_feat_n.T          # [chunk, n_train]
            top_idx = sim.argmax(1); top_sim = sim[np.arange(sim.shape[0]), top_idx]
            best_sim[i:i + CHUNK] = top_sim
            hit = top_sim >= TRAIN_TEST_DUP_COS_TH
            override_label[i:i + CHUNK][hit] = train_labels_arr[top_idx[hit]]
        n_overridden = int((override_label >= 0).sum())
        print(f"[DEDUP-LEAK] test images with cosine>={TRAIN_TEST_DUP_COS_TH} match in train: "
              f"{n_overridden} / {len(test_feat_n)} (max sim seen: {best_sim.max():.4f})")
        if n_overridden > 0:
            print("[DEDUP-LEAK] overriding those predictions with the matched train label "
                  "(free points if this leakage is real; harmless no-op otherwise).")

    if ENSEMBLE_WITH_CONVNEXT and convnext_scores is not None:
        # fix D: average dinov2 test softmax with a convnext test pass at the logit/softmax level.
        _m = build_convnext(); _, vtf = get_tf(_m); del _m; gc.collect(); torch.cuda.empty_cache()
        tl_cnx = DataLoader(ImgOnlyDS(pd.DataFrame({"path": tpaths}), vtf, lab=False), batch_size=BATCH * 2,
                            shuffle=False, num_workers=WORKERS, pin_memory=True, worker_init_fn=seed_worker)
        @torch.no_grad()
        def predict_convnext(model):
            model.eval(); out = []
            for x, _ in tl_cnx:
                x = x.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    p = model(x).softmax(1) + model(torch.flip(x, dims=[3])).softmax(1)
                out.append((p / 2).float().cpu().numpy())
            return np.concatenate(out)
        cnx_tprobs = np.zeros((len(ids), NUM_CLASSES), np.float32); cnx_nu = 0
        for fold in FOLDS_TO_RUN_CONVNEXT:
            cp = f"{WORK}/ckpt_f{fold}.pt"
            if not os.path.exists(cp): continue
            model = build_convnext(); model.load_state_dict(torch.load(cp, map_location=DEVICE)["state_dict"])
            cnx_tprobs += predict_convnext(model); cnx_nu += 1; del model; gc.collect(); torch.cuda.empty_cache()
        if cnx_nu > 0:
            cnx_tprobs /= cnx_nu
            CONVNEXT_ENSEMBLE_WEIGHT = 0.25   # small weight: convnext is a diversity partner, not co-equal
            tprobs = (1 - CONVNEXT_ENSEMBLE_WEIGHT) * tprobs + CONVNEXT_ENSEMBLE_WEIGHT * cnx_tprobs
            print(f"[ensemble] blended convnext (weight={CONVNEXT_ENSEMBLE_WEIGHT}) into dinov2 test probs")

    backbone_results["dinov2"]["test_probs"] = tprobs

    # --- test-time inference for the extra frozen-probe backbones (convnext_small / ...) ---
    # Simple single-scale + hflip TTA (2 views) here rather than the full multi-scale TTA above: these
    # are diversity partners, not the main lever, and keeping their inference cheap protects the <1h budget.
    for name in BACKBONE_NAMES:
        if name == "dinov2":
            continue
        res = backbone_results[name]
        lap(f"[{name}] extracting TEST features (+hflip TTA)")
        b_cfg = res["cfg"]; b_pooling = b_cfg.get("pooling", "mean") if b_cfg["kind"] == "token" else "gap"
        b_batch = b_cfg.get("batch", BATCH_EXTRACT)
        views_b = [cached_extract_features(name, b_cfg["timm_id"], b_cfg["img"], PREPROCESS, b_pooling, "test",
                                           tpaths, res["feat_model"], res["tf"], batch=b_batch, use_mask=res["use_mask"],
                                           mask_mode=PREPROCESS, mask_img=b_cfg["img"])]
        views_b.append(cached_extract_features(name, b_cfg["timm_id"], b_cfg["img"], PREPROCESS, b_pooling, "test",
                                               tpaths, res["feat_model"], res["tf"], flip=True, batch=b_batch,
                                               use_mask=res["use_mask"], mask_mode=PREPROCESS, mask_img=b_cfg["img"]))
        tprobs_b = np.zeros((len(ids), NUM_CLASSES), np.float32); nu_b = 0
        for fold in FOLDS_TO_RUN_DINOV2:
            cp = f"{WORK}/head_f{fold}_{name}.pt"
            if not os.path.exists(cp): continue
            ckpt = torch.load(cp, map_location=DEVICE)
            head = DualHead(ckpt["in_dim"], ckpt.get("hidden", DINOV2_HEAD_HIDDEN), NUM_CLASSES).to(DEVICE)
            head.load_state_dict(ckpt["state_dict"]); head.eval()
            with torch.no_grad():
                for v in views_b:
                    vt = torch.tensor(v, dtype=torch.float32, device=DEVICE)
                    tprobs_b += head(vt).softmax(1).cpu().numpy()
                    nu_b += 1
        backbone_results[name]["test_probs"] = tprobs_b / max(nu_b, 1)
        lap(f"[{name}] TEST inference done")

else:  # convnext
    _m = build_convnext(); _, vtf = get_tf(_m); del _m; gc.collect(); torch.cuda.empty_cache()
    tl = DataLoader(ImgOnlyDS(pd.DataFrame({"path": tpaths}), vtf, lab=False), batch_size=BATCH * 2,
                    shuffle=False, num_workers=WORKERS, pin_memory=True, worker_init_fn=seed_worker)
    @torch.no_grad()
    def predict(model):
        model.eval(); out = []
        for x, _ in tl:
            x = x.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda"):
                p = model(x).softmax(1) + model(torch.flip(x, dims=[3])).softmax(1)
            out.append((p / 2).float().cpu().numpy())
        return np.concatenate(out)
    tprobs = np.zeros((len(ids), NUM_CLASSES), np.float32); nu = 0
    for fold in FOLDS_TO_RUN_CONVNEXT:
        cp = f"{WORK}/ckpt_f{fold}.pt"
        if not os.path.exists(cp): continue
        model = build_convnext(); model.load_state_dict(torch.load(cp, map_location=DEVICE)["state_dict"])
        tprobs += predict(model); nu += 1; del model; gc.collect(); torch.cuda.empty_cache()
    tprobs /= max(nu, 1)
    backbone_results["convnext"]["test_probs"] = tprobs


## 9. Submission

Combines the per-backbone test probabilities with the OOF-chosen ensemble method, applies the class-bias and
(no-op-safe) kNN blend, applies the exact-dup leakage override, writes `submission.csv`, and reports the
train-vs-test predicted-label distribution sanity check. **Verbatim from the original single cell.**


In [ ]:
# ============================================================
# 9. SUBMISSION (+ train-vs-test distribution sanity check)
# ============================================================
P_test_list = [backbone_results[n]["test_probs"] for n in BACKBONE_NAMES]
np.savez(f"{WORK}/test_probs_per_backbone.npz", **{n: backbone_results[n]["test_probs"] for n in BACKBONE_NAMES})
final_test_probs = ensemble_combine(P_test_list)
np.save(f"{WORK}/test_probs.npy", final_test_probs)
final_test_probs_biased = final_test_probs * class_bias
# fix N: kNN gated retrieval blend (no-op if RUN_KNN_RETRIEVAL=False or tuned KNN_ALPHA==0); applied
# BEFORE argmax. The cos>=0.98 exact-dup override further below still runs LAST, taking priority.
if RUN_KNN_RETRIEVAL and KNN_ALPHA > 0:
    knn_test, knn_test_sim = knn_probs(test_feats, feats_all, labels_all)
    _blended = knn_gated_blend(final_test_probs_biased, knn_test, knn_test_sim, KNN_ALPHA)
    _flip = _blended.argmax(1) != final_test_probs_biased.argmax(1)
    print(f"[KNN] test blend alpha={KNN_ALPHA:.2f}: {int(_flip.sum())}/{len(_blended)} predictions changed"
          f"{f' (flipped rows mean top-sim={knn_test_sim[_flip].mean():.3f})' if _flip.any() else ''}")
    final_test_probs_biased = _blended
np.save(f"{WORK}/test_probs_biased.npy", final_test_probs_biased)

sub["predicted"] = final_test_probs_biased.argmax(1).astype(int)
if MODE == "dinov2_linear" and RUN_TRAIN_TEST_DEDUP_CHECK and "override_label" in globals():
    hit = override_label >= 0
    if hit.any():
        sub.loc[hit, "predicted"] = override_label[hit]
        print(f"[DEDUP-LEAK] applied {int(hit.sum())} train-label overrides to final submission")
sub.to_csv(f"{WORK}/submission.csv", index=False)

train_dist = df.label.value_counts(normalize=True).sort_index().reindex(range(NUM_CLASSES), fill_value=0).values
test_dist = pd.Series(sub["predicted"]).value_counts(normalize=True).sort_index().reindex(range(NUM_CLASSES), fill_value=0).values
print("submission dist (count):", sub["predicted"].value_counts().sort_index().tolist())
print("train label proportion :", np.round(train_dist, 3).tolist())
print("test  pred  proportion :", np.round(test_dist, 3).tolist())
if np.abs(train_dist - test_dist).max() > 0.15:
    print("[WARN] test pred distribution diverges >15pp from train -- check domain shift / weight over-correction.")

lap("ALL DONE")
print(f"MODE={MODE} | backbones={BACKBONE_NAMES} | ensemble_method={chosen_method if MODE=='dinov2_linear' or len(BACKBONE_NAMES)>1 else 'n/a (single model)'} "
      f"| IMG={IMG} | PREPROCESS={PREPROCESS} | TTA={USE_TTA} | total wall time={time.time()-T0:.0f}s")
print("Outputs: submission.csv, test_probs.npy, test_probs_per_backbone.npz, oof.npy, decision_weights.npy, "
      f"{'head_f*_<backbone>.pt + dinov2_*_feats.npy + suspect_labels.csv' if MODE=='dinov2_linear' else 'ckpt_f*.pt'}")

## 10. Post-inference Analysis & Misclassifications (NEW)

**New, read-only** analysis that runs *after* the submission is written, so the test artifacts
(`test_feats`, `final_test_probs_biased`, `sub`, `override_label`) all exist and every panel renders fully.


### 10.1 Test Prediction Distribution & Confidence

In [ ]:
# 10.1 Test prediction distribution + confidence (uses the shipped `sub` + final biased probs)
import matplotlib.pyplot as plt
pred_counts = pd.Series(sub["predicted"]).value_counts().sort_index().reindex(range(NUM_CLASSES), fill_value=0)
train_prop = df.label.value_counts(normalize=True).sort_index().reindex(range(NUM_CLASSES), fill_value=0)
test_prop = pred_counts / max(pred_counts.sum(), 1)
display(pd.DataFrame({"class": [CLASS_NAMES[c] for c in range(NUM_CLASSES)],
                      "test_pred_count": pred_counts.values,
                      "test_pred_prop": test_prop.values.round(3),
                      "train_prop": train_prop.values.round(3)}))
test_conf = final_test_probs_biased.max(1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(NUM_CLASSES); w = 0.38
axes[0].bar(x - w / 2, train_prop.values, w, label="train", color="#4C72B0")
axes[0].bar(x + w / 2, test_prop.values, w, label="test pred", color="#DD8452")
axes[0].set_xticks(x); axes[0].set_xticklabels([CLASS_NAMES[c] for c in range(NUM_CLASSES)])
axes[0].set_title("Label distribution: train vs test predictions"); axes[0].legend()
axes[1].hist(test_conf, bins=40, color="#55A868")
axes[1].set_xlabel("max prob (final biased)"); axes[1].set_title("Test prediction confidence")
plt.tight_layout(); plt.savefig(f"{WORK}/test_pred_distribution.png", dpi=120); plt.show()
print(f"Mean test confidence: {test_conf.mean():.4f} | conf<0.5: {int((test_conf < 0.5).sum())} | conf<0.9: {int((test_conf < 0.9).sum())}")


### 10.2 Most / Least Confident Test Predictions

In [ ]:
# 10.2 Most / least confident test predictions (visual sanity check of the shipped labels)
test_pred = final_test_probs_biased.argmax(1)
test_conf = final_test_probs_biased.max(1)
order = np.argsort(-test_conf)
most, least = order[:6], order[::-1][:6]
show_gallery(list(zip([tpaths[i] for i in most],
                      [f"{CLASS_NAMES[test_pred[i]]} ({test_conf[i]:.2f})" for i in most])),
             title="Most confident test predictions")
show_gallery(list(zip([tpaths[i] for i in least],
                      [f"{CLASS_NAMES[test_pred[i]]} ({test_conf[i]:.2f})" for i in least])),
             title="Least confident test predictions")


### 10.3 Train vs Test Embedding Overlap (PCA)

In [ ]:
# 10.3 Train vs test embedding overlap (DINOv2 features, shared PCA basis)
from sklearn.decomposition import PCA
if "feats_all" in globals() and "test_feats" in globals():
    pca_tt = PCA(n_components=2, random_state=SEED).fit(feats_all)
    emb_tr, emb_te = pca_tt.transform(feats_all), pca_tt.transform(test_feats)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(emb_tr[:, 0], emb_tr[:, 1], s=4, alpha=0.35, label="train", color="#4C72B0")
    ax.scatter(emb_te[:, 0], emb_te[:, 1], s=6, alpha=0.55, label="test", color="#C44E52")
    ax.legend(); ax.set_title("Train vs test embedding overlap (DINOv2, PCA)")
    plt.tight_layout(); plt.savefig(f"{WORK}/embedding_train_vs_test.png", dpi=120); plt.show()
else:
    print("[note] needs feats_all (section 5) and test_feats (section 8).")


### 10.4 Domain Shift Analysis

In [ ]:
if "feats_all" in globals() and feats_all is not None and os.path.exists(TEST) and "test_feats" in globals():
    train_feat_n = feats_all / np.clip(np.linalg.norm(feats_all, axis=1, keepdims=True), 1e-8, None)
    test_feat_n = test_feats / np.clip(np.linalg.norm(test_feats, axis=1, keepdims=True), 1e-8, None)

    centroids = np.stack([train_feat_n[df.label.values == c].mean(0) for c in range(NUM_CLASSES)])
    centroids /= np.clip(np.linalg.norm(centroids, axis=1, keepdims=True), 1e-8, None)

    train_sim_to_centroid = train_feat_n @ centroids.T   # [N_train, n_classes]
    test_sim_to_centroid = test_feat_n @ centroids.T     # [N_test, n_classes]

    print("Mean cosine-similarity of TRAIN embeddings to their OWN class centroid:",
          round(float(train_sim_to_centroid[np.arange(len(df)), df.label.values].mean()), 4))
    print("Mean MAX cosine-similarity of TEST embeddings to ANY class centroid  :",
          round(float(test_sim_to_centroid.max(1).mean()), 4),
          "(lower => test sits further from all train class centroids -> possible domain shift)")

    fig, ax = plt.subplots(figsize=(7,4))
    ax.hist(train_sim_to_centroid[np.arange(len(df)), df.label.values], bins=30, alpha=0.5, label="train (own-class sim)")
    ax.hist(test_sim_to_centroid.max(1), bins=30, alpha=0.5, label="test (best-class sim)")
    ax.set_xlabel("cosine similarity to nearest class centroid"); ax.legend()
    ax.set_title("Domain shift proxy: embedding closeness to class centroids")
    plt.tight_layout(); plt.savefig(f"{WORK}/domain_shift.png", dpi=120); plt.show()
else:
    print("[placeholder] Needs `feats_all` (section 5) AND `test_feats` (section 8, Prediction) in "
          "memory. Run sections 4-6 then section 8, then re-run this cell.")


### 10.5 Per-backbone Test Agreement & Leakage Overrides

In [ ]:
# 10.5 Per-backbone test-prediction agreement + exact-dup leakage override summary
names = [n for n in backbone_results if "test_probs" in backbone_results[n]]
tp = {n: backbone_results[n]["test_probs"].argmax(1) for n in names}
if len(tp) >= 2:
    K = len(names); ag = np.zeros((K, K))
    for a in range(K):
        for b in range(K):
            ag[a, b] = float((tp[names[a]] == tp[names[b]]).mean())
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(ag, vmin=0.9, vmax=1.0, cmap="viridis")
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
    ax.set_xticklabels(names, rotation=30, ha="right"); ax.set_yticklabels(names)
    for a in range(K):
        for b in range(K):
            ax.text(b, a, f"{ag[a, b]:.3f}", ha="center", va="center", color="w", fontsize=8)
    ax.set_title("Test-prediction agreement between backbones"); fig.colorbar(im, fraction=0.046)
    plt.tight_layout(); plt.savefig(f"{WORK}/backbone_test_agreement.png", dpi=120); plt.show()
else:
    print("[note] fewer than 2 backbones with test_probs.")

if "override_label" in globals():
    n_ovr = int((override_label >= 0).sum())
    print(f"Train<->test exact-dup leakage overrides applied: {n_ovr} / {len(override_label)}")
    if n_ovr:
        display(pd.Series([CLASS_NAMES[c] for c in override_label[override_label >= 0]]).value_counts()
                .rename("overridden_to").reset_index())


### 10.6 Misclassified Validation Samples

In [ ]:
# 10.6 Galleries of OOF-misclassified validation images (true -> predicted (confidence))
if len(wrong):
    w_sorted = wrong.sort_values("conf", ascending=False)
    n_show = min(36, len(w_sorted))
    for start in range(0, n_show, 6):
        b = w_sorted.iloc[start:start + 6]
        show_gallery(list(zip(b["path"], b["true_name"] + "->" + b["pred_name"] + " (" + b["conf"].round(2).astype(str) + ")")),
                     title=f"Misclassified val samples {start + 1}-{start + len(b)}")
else:
    print("No OOF misclassifications to display (every scored OOF row was correct).")
